In [ ]:
# Dataset-acquisition setup only: this does not train or download files.
from pathlib import Path
import shutil, platform
from google.colab import drive

drive.mount('/content/drive')
PROJECT = Path('/content/drive/MyDrive/Persistent Visual-to-MANO Hand Tracker')
STAGING = PROJECT / '04_DATASET_STAGING'
FOLDERS = [
    STAGING / 'ReInterHand' / '00_archives',
    STAGING / 'ReInterHand' / '01_extracted',
    STAGING / 'ReInterHand' / '02_checksums',
    STAGING / 'InterHand2.6M' / '00_archives',
    STAGING / 'InterHand2.6M' / '01_extracted',
    STAGING / 'InterHand2.6M' / '02_annotations',
    STAGING / 'InterHand2.6M' / '03_checksums',
    STAGING / '_transfer_logs',
]
missing = [str(p) for p in FOLDERS if not p.parent.exists()]
if not PROJECT.exists():
    raise FileNotFoundError(f'Expected project folder not found: {PROJECT}')
for folder in FOLDERS:
    folder.mkdir(parents=True, exist_ok=True)
usage = shutil.disk_usage('/content')
print('Drive mounted; project folder verified:', PROJECT)
print('Dataset staging folders ready:', len(FOLDERS))
print(f'Colab local disk free: {usage.free / (1024**3):.1f} GiB (temporary; do not stage the whole dataset here)')
print('Runtime:', platform.platform())
print('No dataset download or training has started.')

In [ ]:
# ReInterHand source + Drive write preflight (metadata only; no training).
# Safe to rerun: complete files are verified by their sidecar SHA-256 and skipped.
from pathlib import Path
import hashlib, os, time
import requests
from google.colab import drive

DRIVE_MOUNT = Path('/content/drive')
if not (DRIVE_MOUNT / 'MyDrive').exists():
    drive.mount('/content/drive')
PROJECT = DRIVE_MOUNT / 'MyDrive' / 'Persistent Visual-to-MANO Hand Tracker'
STAGING = PROJECT / '04_DATASET_STAGING'
BASE_URL = 'https://fb-baas-f32eacb9-8abb-11eb-b2b8-4857dd089e15.s3.amazonaws.com/ReInterHand'
CAPTURES = [
    'm--20210701--1058--0000000--pilot--relightablehandsy--participant0--two-hands',
    'm--20220628--1327--BKS383--pilot--ProjectGoliath--ContinuousHandsy--two-hands',
    'm--20221007--1215--HIR112--pilot--ProjectGoliathScript--Hands--two-hands',
    'm--20221110--1033--TQH976--pilot--ProjectGoliathScript--Hands--two-hands',
    'm--20221111--0944--JFQ550--pilot--ProjectGoliathScript--Hands--two-hands',
    'm--20221215--0949--RNS217--pilot--ProjectGoliathScript--Hands--two-hands',
    'm--20221216--0953--NKC880--pilot--ProjectGoliathScript--Hands--two-hands',
    'm--20230313--1433--TXB805--pilot--ProjectGoliath--Hands--two-hands',
    'm--20230317--1130--QZX685--pilot--ProjectGoliath--Hands--two-hands',
    'm--20230317--1433--TRO760--pilot--ProjectGoliath--Hands--two-hands',
]
if not PROJECT.is_dir():
    raise FileNotFoundError(f'Project folder missing on mounted Drive: {PROJECT}')

def sha256_file(path):
    h = hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda: f.read(4 * 1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def fetch_small(url, dest):
    dest.parent.mkdir(parents=True, exist_ok=True)
    sidecar = dest.with_name(dest.name + '.sha256')
    if dest.is_file() and sidecar.is_file():
        expected = sidecar.read_text().strip().split()[0]
        if sha256_file(dest) == expected:
            return 'verified-existing'
    tmp = dest.with_name(dest.name + '.partial')
    last_error = None
    for attempt in range(1, 5):
        try:
            with requests.get(url, stream=True, timeout=(20, 90)) as response:
                response.raise_for_status()
                digest = hashlib.sha256()
                written = 0
                with tmp.open('wb') as out:
                    for chunk in response.iter_content(chunk_size=1024 * 1024):
                        if chunk:
                            out.write(chunk)
                            digest.update(chunk)
                            written += len(chunk)
                    out.flush()
                    os.fsync(out.fileno())
            if written == 0:
                raise IOError('download returned an empty file')
            os.replace(tmp, dest)
            sidecar.write_text(f'{digest.hexdigest()}  {dest.name}  {written} bytes\n')
            return f'downloaded {written:,} bytes'
        except Exception as exc:
            last_error = exc
            try:
                tmp.unlink(missing_ok=True)
            except Exception:
                pass
            if attempt < 4:
                time.sleep(attempt * 2)
    raise RuntimeError(f'Failed after 4 attempts: {url}') from last_error

jobs = []
for capture in CAPTURES:
    meta_dir = STAGING / 'ReInterHand' / '02_checksums' / capture
    for filename in ('CHECKSUM', 'frame_list.txt', 'frame_list_orig.txt'):
        jobs.append((f'{BASE_URL}/{capture}/{filename}', meta_dir / filename))
    cam_dir = STAGING / 'ReInterHand' / '00_archives' / capture / 'metadata' / 'Mugsy_cameras'
    jobs.append((f'{BASE_URL}/{capture}/Mugsy_cameras/cam_params.json', cam_dir / 'cam_params.json'))

print(f'Checking official ReInterHand endpoint and Drive writes: {len(jobs)} small files')
started = time.time()
for n, (url, path) in enumerate(jobs, 1):
    status = fetch_small(url, path)
    print(f'[{n:02d}/{len(jobs)}] {path.name}: {status}', flush=True)
print(f'Preflight complete in {time.time()-started:.1f}s; files are on Drive under {STAGING / "ReInterHand"}.')

In [ ]:
# Resumable ReInterHand Mugsy 30-fps archive transfer directly to Google Drive.
# Safe to rerun: completed shards are skipped; interrupted .partial files resume with HTTP Range.
import ast, hashlib, json, os, time, datetime
from pathlib import Path
import requests

PROJECT = Path('/content/drive/MyDrive/Persistent Visual-to-MANO Hand Tracker')
STAGING = PROJECT / '04_DATASET_STAGING'
if not Path('/content/drive/MyDrive').exists():
    from google.colab import drive
    drive.mount('/content/drive')
if not PROJECT.exists():
    raise FileNotFoundError(f'Drive project folder not found: {PROJECT}')

# Transfer controls: bounded memory, Drive-only large files, resumable partials.
BATCH_LIMIT = 0        # PAUSED after HIR112/ac; do not resume acquisition until user instructs
MAX_RUN_SECONDS = 60 * 60        # stop after one hour even if network is slow
CHUNK_SIZE = 4 * 1024**2         # 4 MiB RAM buffer
HEARTBEAT_SECONDS = 20
MAX_RETRIES = 5

SOURCE_SCRIPT = 'https://raw.githubusercontent.com/facebookresearch/InterWild/main/tool/ReInterHand/download/download_mugsy_per_segment.py'
S3_ROOT = 'https://fb-baas-f32eacb9-8abb-11eb-b2b8-4857dd089e15.s3.amazonaws.com/ReInterHand'
source_resp = requests.get(SOURCE_SCRIPT, timeout=(20, 90))
source_resp.raise_for_status()
source_text = source_resp.text
source_file = STAGING / 'ReInterHand' / '00_archives' / '_source' / 'download_mugsy_per_segment.py'
source_file.parent.mkdir(parents=True, exist_ok=True)
source_file.write_text(source_text, encoding='utf-8')
source_sha = hashlib.sha256(source_text.encode('utf-8')).hexdigest()
source_file.with_suffix(source_file.suffix + '.sha256').write_text(source_sha + '  ' + source_file.name + '\n', encoding='utf-8')

# Parse only literal capture/suffix lists from the official script; do not execute remote code.
tree = ast.parse(source_text)
literals = {}
for node in tree.body:
    if isinstance(node, ast.Assign):
        for target in node.targets:
            if isinstance(target, ast.Name) and target.id in ('capture_id_list', 'name_list'):
                literals[target.id] = ast.literal_eval(node.value)
if not isinstance(literals.get('capture_id_list'), list) or not isinstance(literals.get('name_list'), dict):
    raise RuntimeError('Official download script did not contain the expected literal capture_id_list/name_list.')
captures = literals['capture_id_list']
suffixes_by_capture = literals['name_list']
if any(c not in suffixes_by_capture for c in captures):
    raise RuntimeError('Official archive suffix map is incomplete; refusing to guess filenames.')
plan = [(cap, suffix) for cap in captures for suffix in suffixes_by_capture[cap]]

archive_root = STAGING / 'ReInterHand' / '00_archives'
log_path = STAGING / '_transfer_logs' / 'reinterhand_mugsy_per_segment.jsonl'
log_path.parent.mkdir(parents=True, exist_ok=True)

def utc_now():
    return datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds')

def log_event(event, **fields):
    entry = {'time_utc': utc_now(), 'event': event, **fields}
    with log_path.open('a', encoding='utf-8') as f:
        f.write(json.dumps(entry, sort_keys=True) + '\n')

def save_meta(path, data):
    meta = path.with_name(path.name + '.transfer.json')
    tmp = meta.with_name(meta.name + '.tmp')
    tmp.write_text(json.dumps(data, indent=2, sort_keys=True), encoding='utf-8')
    tmp.replace(meta)

def sha256_file(path):
    h = hashlib.sha256()
    size = 0
    last_report = time.monotonic()
    with path.open('rb') as f:
        while True:
            block = f.read(8 * 1024**2)
            if not block:
                break
            h.update(block)
            size += len(block)
            if time.monotonic() - last_report >= HEARTBEAT_SECONDS:
                print(f'  hashing {path.name}: {size / 1024**3:.2f} GiB read', flush=True)
                last_report = time.monotonic()
    return h.hexdigest(), size

session = requests.Session()
started = time.monotonic()
received = 0
completed_now = 0
skipped = 0
failed = None
last_heartbeat = started
print(f'Official source: {SOURCE_SCRIPT}', flush=True)
print(f'Captures: {len(captures)}; total Mugsy per-segment shards in official list: {len(plan)}', flush=True)
print(f'Drive target: {archive_root}', flush=True)
print(f'Run cap: {BATCH_LIMIT / 1024**3:.1f} GiB or {MAX_RUN_SECONDS / 60:.0f} minutes; partials stay on Drive.', flush=True)
log_event('batch_start', captures=len(captures), planned_shards=len(plan), batch_limit_bytes=BATCH_LIMIT, source_sha256=source_sha)

for shard_number, (capture, suffix) in enumerate(plan, 1):
    filename = 'envmap_per_segment.tar.gz' + suffix
    url = f'{S3_ROOT}/{capture}/Mugsy_cameras/{filename}'
    target_dir = archive_root / capture / 'Mugsy_cameras'
    target_dir.mkdir(parents=True, exist_ok=True)
    final_path = target_dir / filename
    partial_path = target_dir / (filename + '.partial')
    meta_path = final_path.with_name(final_path.name + '.transfer.json')

    if final_path.is_file() and meta_path.is_file():
        try:
            old_meta = json.loads(meta_path.read_text(encoding='utf-8'))
            if old_meta.get('status') == 'complete' and old_meta.get('bytes') == final_path.stat().st_size and old_meta.get('sha256'):
                skipped += 1
                continue
        except Exception:
            pass

    if received >= BATCH_LIMIT or time.monotonic() - started >= MAX_RUN_SECONDS:
        break

    try:
        head = session.head(url, allow_redirects=True, timeout=(20, 90))
        head.raise_for_status()
        remote_size = int(head.headers['Content-Length']) if head.headers.get('Content-Length', '').isdigit() else None
        etag = head.headers.get('ETag')
    except Exception as exc:
        failed = f'HEAD failed for {filename}: {type(exc).__name__}: {exc}'
        print(f'ERROR: {failed}', flush=True)
        log_event('head_error', capture=capture, suffix=suffix, error=failed)
        break

    if final_path.exists():
        if remote_size is not None and final_path.stat().st_size == remote_size:
            print(f'Existing complete-size archive; hashing to record it: {filename}', flush=True)
            digest, actual_size = sha256_file(final_path)
            save_meta(final_path, {'status': 'complete', 'url': url, 'bytes': actual_size, 'sha256': digest, 'etag': etag, 'verified_against_remote_checksum': False, 'recorded_utc': utc_now()})
            skipped += 1
            continue
        failed = f'Existing destination conflicts with remote size for {filename}; left untouched.'
        print(f'ERROR: {failed}', flush=True)
        log_event('destination_conflict', capture=capture, suffix=suffix, remote_bytes=remote_size, local_bytes=final_path.stat().st_size)
        break

    offset = partial_path.stat().st_size if partial_path.exists() else 0
    if remote_size is not None and offset > remote_size:
        stale = partial_path.with_name(partial_path.name + '.stale.' + datetime.datetime.now().strftime('%Y%m%d%H%M%S'))
        partial_path.replace(stale)
        print(f'Quarantined oversized partial {stale.name}; restarting this shard.', flush=True)
        offset = 0
    if remote_size is not None and offset == remote_size and offset > 0:
        print(f'Finalizing previously complete partial: {filename}', flush=True)
        digest, actual_size = sha256_file(partial_path)
        partial_path.replace(final_path)
        save_meta(final_path, {'status': 'complete', 'url': url, 'bytes': actual_size, 'sha256': digest, 'etag': etag, 'verified_against_remote_checksum': False, 'recorded_utc': utc_now()})
        completed_now += 1
        log_event('shard_complete', capture=capture, suffix=suffix, bytes=actual_size, sha256=digest, etag=etag)
        continue

    print(f'[{shard_number}/{len(plan)}] {capture} / {filename} | remote={remote_size if remote_size is not None else "unknown"} bytes | resume={offset} bytes', flush=True)
    attempts = 0
    shard_done = False
    while attempts < MAX_RETRIES and not shard_done:
        if received >= BATCH_LIMIT or time.monotonic() - started >= MAX_RUN_SECONDS:
            break
        attempts += 1
        offset = partial_path.stat().st_size if partial_path.exists() else 0
        headers = {'Range': f'bytes={offset}-'} if offset else {}
        try:
            with session.get(url, headers=headers, stream=True, timeout=(20, 120)) as response:
                if offset and response.status_code == 416:
                    if remote_size is not None and offset == remote_size:
                        shard_done = True
                        break
                    raise RuntimeError(f'HTTP 416 at offset {offset} of {remote_size}')
                response.raise_for_status()
                if offset:
                    if response.status_code == 206:
                        cr = response.headers.get('Content-Range', '')
                        if not cr.startswith(f'bytes {offset}-'):
                            raise RuntimeError(f'Unexpected Content-Range {cr!r} for resume offset {offset}')
                        mode = 'ab'
                    elif response.status_code == 200:
                        # Server ignored Range: safely restart from byte zero.
                        offset = 0
                        mode = 'wb'
                    else:
                        raise RuntimeError(f'Unexpected resume response HTTP {response.status_code}')
                else:
                    if response.status_code not in (200, 206):
                        raise RuntimeError(f'Unexpected initial response HTTP {response.status_code}')
                    mode = 'wb'
                current_total = remote_size
                if response.status_code == 206 and current_total is None:
                    cr = response.headers.get('Content-Range', '')
                    if '/' in cr and cr.rsplit('/', 1)[1].isdigit():
                        current_total = int(cr.rsplit('/', 1)[1])
                last_heartbeat = time.monotonic()
                with partial_path.open(mode) as out:
                    for block in response.iter_content(chunk_size=CHUNK_SIZE):
                        if not block:
                            continue
                        budget_left = BATCH_LIMIT - received
                        if budget_left <= 0:
                            break
                        if len(block) > budget_left:
                            block = block[:budget_left]
                        out.write(block)
                        out.flush()
                        received += len(block)
                        local_now = partial_path.stat().st_size
                        now = time.monotonic()
                        if now - last_heartbeat >= HEARTBEAT_SECONDS or received >= BATCH_LIMIT:
                            elapsed = max(now - started, 1e-3)
                            speed = received / elapsed / 1024**2
                            total_text = f'/{current_total / 1024**3:.2f} GiB' if current_total else ''
                            print(f'  progress: run={received / 1024**3:.2f} GiB; shard={local_now / 1024**3:.2f}{total_text}; avg={speed:.1f} MiB/s; elapsed={elapsed / 60:.1f} min', flush=True)
                            log_event('progress', capture=capture, suffix=suffix, run_bytes=received, shard_bytes=local_now, remote_bytes=current_total, avg_mib_s=round(speed, 2))
                            last_heartbeat = now
                        if received >= BATCH_LIMIT or time.monotonic() - started >= MAX_RUN_SECONDS:
                            break
                local_now = partial_path.stat().st_size if partial_path.exists() else 0
                if current_total is not None and local_now == current_total:
                    shard_done = True
                elif current_total is None and response.ok and received < BATCH_LIMIT and time.monotonic() - started < MAX_RUN_SECONDS:
                    # Clean HTTP stream end with no advertised size means the shard completed.
                    shard_done = True
                elif received >= BATCH_LIMIT or time.monotonic() - started >= MAX_RUN_SECONDS:
                    break
                else:
                    raise RuntimeError(f'Stream ended at {local_now} bytes; expected {current_total}')
        except Exception as exc:
            print(f'  attempt {attempts}/{MAX_RETRIES} interrupted: {type(exc).__name__}: {exc}; partial preserved.', flush=True)
            log_event('attempt_error', capture=capture, suffix=suffix, attempt=attempts, error=f'{type(exc).__name__}: {exc}', partial_bytes=partial_path.stat().st_size if partial_path.exists() else 0)
            if attempts < MAX_RETRIES and received < BATCH_LIMIT and time.monotonic() - started < MAX_RUN_SECONDS:
                time.sleep(min(2 ** attempts, 20))
            else:
                failed = f'Download failed for {filename} after {attempts} attempts.'

    if shard_done:
        digest, actual_size = sha256_file(partial_path)
        partial_path.replace(final_path)
        save_meta(final_path, {'status': 'complete', 'url': url, 'bytes': actual_size, 'sha256': digest, 'etag': etag, 'verified_against_remote_checksum': False, 'recorded_utc': utc_now(), 'official_script_sha256': source_sha})
        completed_now += 1
        print(f'  COMPLETE: {filename} | {actual_size / 1024**3:.2f} GiB | SHA256 {digest}', flush=True)
        log_event('shard_complete', capture=capture, suffix=suffix, bytes=actual_size, sha256=digest, etag=etag)
    elif failed:
        break
    else:
        print(f'  PAUSED safely: {filename} partial is {partial_path.stat().st_size / 1024**3:.2f} GiB and remains on Drive.', flush=True)
        log_event('shard_paused', capture=capture, suffix=suffix, partial_bytes=partial_path.stat().st_size if partial_path.exists() else 0)
        break

    if failed or received >= BATCH_LIMIT or time.monotonic() - started >= MAX_RUN_SECONDS:
        break

elapsed = time.monotonic() - started
partial_bytes = sum(p.stat().st_size for p in archive_root.rglob('*.partial'))
summary = {
    'time_utc': utc_now(), 'category': 'ReInterHand/Mugsy_cameras/envmap_per_segment',
    'new_bytes_received_this_run': received, 'new_gib_received_this_run': round(received / 1024**3, 3),
    'completed_shards_this_run': completed_now, 'already_completed_shards_skipped': skipped,
    'partial_bytes_remaining_across_this_category': partial_bytes, 'official_shards_total': len(plan),
    'elapsed_seconds': round(elapsed, 1), 'error': failed,
    'destination': str(archive_root), 'source_script_sha256': source_sha,
}
summary_path = STAGING / '_transfer_logs' / 'reinterhand_mugsy_per_segment_latest.json'
summary_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')
log_event('batch_end', **summary)
print('\nTRANSFER BATCH SUMMARY', flush=True)
print(json.dumps(summary, indent=2), flush=True)
if received >= BATCH_LIMIT:
    print('Batch cap reached. Rerun this same cell to continue; it will resume partials and skip verified-complete shards.', flush=True)
elif failed:
    print('Stopped at the reported source/network issue; partial data remains on Drive for a safe retry.', flush=True)
else:
    print('Run stopped at its time cap or completed the available plan; rerun this same cell to continue if shards remain.', flush=True)


In [ ]:
from pathlib import Path
p=Path('/content/drive/MyDrive/Persistent Visual-to-MANO Hand Tracker/04_DATASET_STAGING/ReInterHand/00_archives/_source/download_mugsy_per_segment.py')
lines=p.read_text().splitlines()
for i,line in enumerate(lines):
    if any(x in line.lower() for x in ('url', 's3', 'mugsy_cameras', 'envmap_per_segment')):
        lo=max(0,i-2); hi=min(len(lines),i+3)
        print(f'--- lines {lo+1}-{hi} ---')
        for j in range(lo,hi): print(f'{j+1}: {lines[j]}')


In [ ]:
from google.colab import drive
import shutil, time

def disk_report(label):
    u = shutil.disk_usage('/content')
    print(f'{label}: runtime disk used={u.used/1024**3:.2f} GiB, free={u.free/1024**3:.2f} GiB', flush=True)

disk_report('Before flush')
print('Flushing and unmounting mounted Drive...', flush=True)
drive.flush_and_unmount()
disk_report('After flush/unmount')
print('Remounting Drive...', flush=True)
drive.mount('/content/drive')
disk_report('After remount')
print('Drive remount check:', ( __import__('pathlib').Path('/content/drive/MyDrive/Persistent Visual-to-MANO Hand Tracker/04_DATASET_STAGING/ReInterHand/00_archives').exists()))

In [ ]:
import json
from pathlib import Path
root = Path('/content/drive/MyDrive/Persistent Visual-to-MANO Hand Tracker/04_DATASET_STAGING/ReInterHand/00_archives')
log = Path('/content/drive/MyDrive/Persistent Visual-to-MANO Hand Tracker/04_DATASET_STAGING/_transfer_logs/reinterhand_mugsy_per_segment.jsonl')
cap = 'm--20221007--1215--HIR112--pilot--ProjectGoliathScript--Hands--two-hands'
name = 'envmap_per_segment.tar.gzac'
final = root / cap / 'Mugsy_cameras' / name
meta_path = final.with_name(final.name + '.transfer.json')
meta = json.loads(meta_path.read_text(encoding='utf-8'))
events = [json.loads(s) for s in log.read_text(encoding='utf-8').splitlines() if s.strip()]
logged = [e for e in events if e.get('event') == 'shard_complete' and e.get('capture') == cap and e.get('suffix') == 'ac']
print('HIR112/ac after Drive remount:')
print('  exists:', final.is_file(), 'size:', final.stat().st_size if final.exists() else None)
print('  sidecar complete:', meta.get('status') == 'complete', 'sidecar bytes match:', meta.get('bytes') == (final.stat().st_size if final.exists() else -1))
print('  sidecar SHA-256:', meta.get('sha256'))
print('  completion log matches:', bool(logged) and logged[-1].get('sha256') == meta.get('sha256') and logged[-1].get('bytes') == meta.get('bytes'))
files = [p for p in root.rglob('envmap_per_segment.tar.gz*') if p.is_file() and not p.name.endswith('.partial')]
print(f'Final archive files currently present: {len(files)}; total bytes: {sum(p.stat().st_size for p in files):,} ({sum(p.stat().st_size for p in files)/1024**3:.2f} GiB)')
partials = [p for p in root.rglob('envmap_per_segment.tar.gz*.partial') if p.is_file()]
print('Partial archives preserved:')
for p in partials: print(' ', p.relative_to(root), p.stat().st_size)
bks = 'm--20220628--1327--BKS383--pilot--ProjectGoliath--ContinuousHandsy--two-hands'
an = root / bks / 'Mugsy_cameras' / 'envmap_per_segment.tar.gzan'
print('BKS383/an completion log exists but final archive is', 'present' if an.exists() else 'missing', '; log event count:', sum(e.get('event') == 'shard_complete' and e.get('capture') == bks and e.get('suffix') == 'an' for e in events))


In [ ]:
import json, re
from pathlib import Path
root=Path('/content/drive/MyDrive/Persistent Visual-to-MANO Hand Tracker/04_DATASET_STAGING/ReInterHand/00_archives')
stem='envmap_per_segment.tar.gz'
by_capture={}
for p in root.rglob(stem+'*'):
    if not p.is_file(): continue
    suffix=p.name[len(stem):]
    if len(suffix)==2 and suffix.isalpha() and suffix.islower():
        cap=p.parents[1].name
        meta=p.with_name(p.name+'.transfer.json')
        ok=False
        if meta.is_file():
            try:
                m=json.loads(meta.read_text())
                ok=m.get('status')=='complete' and m.get('bytes')==p.stat().st_size and bool(m.get('sha256'))
            except Exception: pass
        row=by_capture.setdefault(cap, {'count':0,'bytes':0,'sidecars_ok':0})
        row['count']+=1; row['bytes']+=p.stat().st_size; row['sidecars_ok']+=int(ok)
for cap,row in sorted(by_capture.items()):
    print(f"{cap}: files={row['count']}, bytes={row['bytes']:,}, GiB={row['bytes']/1024**3:.2f}, complete_sidecars={row['sidecars_ok']}")
print('TOTAL finalized shards:',sum(r['count'] for r in by_capture.values()),'bytes:',sum(r['bytes'] for r in by_capture.values()),'GiB:',round(sum(r['bytes'] for r in by_capture.values())/1024**3,2))
print('partials:')
for p in sorted(root.rglob(stem+'*.partial')):
    if p.is_file(): print(p.relative_to(root),p.stat().st_size)


In [ ]:
# Finish exactly HIR112 shard ac; this cell does not iterate over any other shard.
import datetime, hashlib, json, os, time
from pathlib import Path
import requests

PROJECT = Path('/content/drive/MyDrive/Persistent Visual-to-MANO Hand Tracker')
STAGING = PROJECT / '04_DATASET_STAGING'
ARCHIVE_ROOT = STAGING / 'ReInterHand' / '00_archives'
LOG_PATH = STAGING / '_transfer_logs' / 'reinterhand_mugsy_per_segment.jsonl'
CAPTURE = 'm--20221007--1215--HIR112--pilot--ProjectGoliathScript--Hands--two-hands'
FILENAME = 'envmap_per_segment.tar.gzac'
URL = f'https://fb-baas-f32eacb9-8abb-11eb-b2b8-4857dd089e15.s3.amazonaws.com/ReInterHand/{CAPTURE}/Mugsy_cameras/{FILENAME}'
TARGET_DIR = ARCHIVE_ROOT / CAPTURE / 'Mugsy_cameras'
FINAL = TARGET_DIR / FILENAME
PARTIAL = TARGET_DIR / (FILENAME + '.partial')
META = FINAL.with_name(FINAL.name + '.transfer.json')
TARGET_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
EXPECTED_BYTES = 2 * 1024**3
CHUNK_SIZE = 4 * 1024**2

def log_event(event, **fields):
    row = {'time_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'), 'event': event, 'capture': CAPTURE, 'suffix': 'ac', **fields}
    with LOG_PATH.open('a', encoding='utf-8') as f:
        f.write(json.dumps(row, sort_keys=True) + '\n')
        f.flush()
        os.fsync(f.fileno())

def sha256_file(path):
    h = hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda: f.read(CHUNK_SIZE), b''):
            h.update(block)
    return h.hexdigest()

if not (Path('/content/drive/MyDrive').exists() and PROJECT.is_dir()):
    raise RuntimeError('Google Drive is not mounted at the expected project path.')

session = requests.Session()
head = session.head(URL, allow_redirects=True, timeout=(20, 90))
head.raise_for_status()
remote_bytes = int(head.headers.get('Content-Length', '0'))
etag = head.headers.get('ETag')
print(f'Target only: HIR112/ac; remote size={remote_bytes:,} bytes; Drive destination={FINAL}', flush=True)
if remote_bytes != EXPECTED_BYTES:
    raise RuntimeError(f'Remote size mismatch: expected {EXPECTED_BYTES}, got {remote_bytes}. No shard written.')

if FINAL.is_file() and META.is_file():
    old = json.loads(META.read_text(encoding='utf-8'))
    if old.get('status') == 'complete' and old.get('bytes') == FINAL.stat().st_size and old.get('sha256'):
        digest = sha256_file(FINAL)
        if digest == old['sha256']:
            print(f'Already complete and SHA-256 verified: {digest}', flush=True)
            raise SystemExit
        raise RuntimeError('Existing completed file failed SHA-256; left untouched.')
if FINAL.exists():
    raise RuntimeError('A final archive exists without a verified complete sidecar; left untouched.')

offset = PARTIAL.stat().st_size if PARTIAL.exists() else 0
if offset > remote_bytes:
    raise RuntimeError(f'Partial exceeds remote size ({offset} > {remote_bytes}); left untouched.')
log_event('shard_start', url=URL, remote_bytes=remote_bytes, resume_bytes=offset, etag=etag, target=str(FINAL))
print(f'Resuming from {offset:,} bytes; bounded buffer={CHUNK_SIZE // 1024**2} MiB.', flush=True)
attempt = 0
last_report = time.monotonic()
while offset < remote_bytes:
    attempt += 1
    if attempt > 10:
        raise RuntimeError('Stopped after 10 interrupted transfer attempts; partial is preserved.')
    headers = {'Range': f'bytes={offset}-'} if offset else {}
    try:
        with session.get(URL, headers=headers, stream=True, timeout=(20, 120)) as response:
            if offset:
                if response.status_code != 206:
                    raise RuntimeError(f'Range resume expected HTTP 206, received {response.status_code}.')
                content_range = response.headers.get('Content-Range', '')
                if not content_range.startswith(f'bytes {offset}-'):
                    raise RuntimeError(f'Unexpected Content-Range: {content_range}')
                mode = 'ab'
            else:
                if response.status_code != 200:
                    raise RuntimeError(f'Initial download expected HTTP 200, received {response.status_code}.')
                mode = 'wb'
            response.raise_for_status()
            with PARTIAL.open(mode) as out:
                for block in response.iter_content(chunk_size=CHUNK_SIZE):
                    if not block:
                        continue
                    out.write(block)
                    now = time.monotonic()
                    if now - last_report >= 20:
                        out.flush()
                        os.fsync(out.fileno())
                        offset = PARTIAL.stat().st_size
                        elapsed = max(now - start_time, 1e-6) if 'start_time' in globals() else 0.0
                        print(f'Progress: {offset / 1024**3:.2f}/{remote_bytes / 1024**3:.2f} GiB', flush=True)
                        log_event('progress', remote_bytes=remote_bytes, shard_bytes=offset, etag=etag)
                        last_report = now
                out.flush()
                os.fsync(out.fileno())
        new_offset = PARTIAL.stat().st_size
        if new_offset <= offset and new_offset < remote_bytes:
            raise IOError('Transfer made no progress; retrying from saved partial.')
        offset = new_offset
    except Exception as exc:
        offset = PARTIAL.stat().st_size if PARTIAL.exists() else 0
        log_event('retry', remote_bytes=remote_bytes, shard_bytes=offset, attempt=attempt, error=f'{type(exc).__name__}: {exc}')
        if attempt >= 10:
            raise
        print(f'Connection interrupted ({type(exc).__name__}); preserving {offset:,} bytes and resuming.', flush=True)
        time.sleep(min(2 * attempt, 15))

if PARTIAL.stat().st_size != remote_bytes:
    raise RuntimeError(f'Final length mismatch: {PARTIAL.stat().st_size} != {remote_bytes}; partial preserved.')
print('Download complete; computing SHA-256 from the saved Drive file...', flush=True)
digest = sha256_file(PARTIAL)
PARTIAL.replace(FINAL)
meta_tmp = META.with_name(META.name + '.tmp')
meta_tmp.write_text(json.dumps({'status': 'complete', 'url': URL, 'bytes': remote_bytes, 'sha256': digest, 'etag': etag, 'verified_against_remote_checksum': False, 'recorded_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds')}, indent=2, sort_keys=True), encoding='utf-8')
meta_tmp.replace(META)
log_event('shard_complete', bytes=remote_bytes, sha256=digest, etag=etag)
print(f'COMPLETE HIR112/ac: {FINAL.stat().st_size:,} bytes; SHA-256 {digest}', flush=True)


In [ ]:
from pathlib import Path
base = Path('/content/drive/MyDrive/Persistent Visual-to-MANO Hand Tracker/04_DATASET_STAGING/ReInterHand')
cap = 'm--20210701--1058--0000000--pilot--relightablehandsy--participant0--two-hands'
checks = base / '02_checksums' / cap / 'CHECKSUM'
archives = base / '00_archives' / cap
extracted = base / '01_extracted' / cap
lines = checks.read_text(encoding='utf-8', errors='replace').splitlines()
print('Supervision archive check for the complete capture:')
for line in lines:
    rel = line.split()[0] if line.split() else ''
    if rel.startswith(('orig_fits/', 'mano_fits/')):
        a = archives / rel
        e = extracted / rel
        print(f'{rel} | archive={a.is_file()} | extracted={e.exists()} | bytes={a.stat().st_size if a.is_file() else 0}')
print('Capture frame/camera metadata:')
for rel in ('frame_list.txt', 'Mugsy_cameras/cam_params.json'):
    found = any((p / rel).is_file() for p in (archives, extracted))
    print(rel, found)

In [ ]:
# Minimal resumable acquisition: finish BKS383/an and fetch MANO + right-keypoint labels for all three captures with staged images.
# Writes large archives directly to Drive, verifies each against official CHECKSUM MD5, and logs every shard.
import hashlib, json, os, re, time, datetime
from pathlib import Path
import requests
from google.colab import drive

if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive')
PROJECT = Path('/content/drive/MyDrive/Persistent Visual-to-MANO Hand Tracker')
STAGING = PROJECT / '04_DATASET_STAGING'
BASE = 'https://fb-baas-f32eacb9-8abb-11eb-b2b8-4857dd089e15.s3.amazonaws.com/ReInterHand'
ARCHIVE_ROOT = STAGING / 'ReInterHand' / '00_archives'
CHECKSUM_ROOT = STAGING / 'ReInterHand' / '02_checksums'
LOG = STAGING / '_transfer_logs' / 'reinterhand_labels_and_resume.jsonl'
CHUNK = 4 * 1024 * 1024
MAX_RUN_SECONDS = 55 * 60
CAPTURES = {
    'm--20210701--1058--0000000--pilot--relightablehandsy--participant0--two-hands': {'mano_fits':['aa'], 'orig_fits':['aa','ab','ac','ad']},
    'm--20220628--1327--BKS383--pilot--ProjectGoliath--ContinuousHandsy--two-hands': {'mano_fits':['aa'], 'orig_fits':['aa','ab']},
    'm--20221007--1215--HIR112--pilot--ProjectGoliathScript--Hands--two-hands': {'mano_fits':['aa'], 'orig_fits':['aa','ab','ac','ad','ae']},
}
# Close the already-started BKS383 image shard; do not resume the general all-captures image loop.
JOBS = [('m--20220628--1327--BKS383--pilot--ProjectGoliath--ContinuousHandsy--two-hands', 'Mugsy_cameras/envmap_per_segment.tar.gzan')]
for cap, groups in CAPTURES.items():
    for folder, suffixes in groups.items():
        stem = 'mano_fits.tar.gz' if folder == 'mano_fits' else 'orig_fits.tar.gz'
        JOBS.extend((cap, f'{folder}/{stem}{suffix}') for suffix in suffixes)

if not PROJECT.is_dir():
    raise FileNotFoundError(f'Drive project folder is missing: {PROJECT}')

def read_checksums(path):
    found = {}
    for line in path.read_text(encoding='utf-8').splitlines():
        fields = line.strip().split()
        if len(fields) < 2:
            continue
        if re.fullmatch(r'[0-9a-fA-F]{32}', fields[-1]):
            rel, digest = ' '.join(fields[:-1]).lstrip('*./'), fields[-1].lower()
        elif re.fullmatch(r'[0-9a-fA-F]{32}', fields[0]):
            rel, digest = ' '.join(fields[1:]).lstrip('*./'), fields[0].lower()
        else:
            continue
        found[rel.replace(chr(92), '/')] = digest
    return found

def checksum_for(table, target):
    matches = [(name, md5) for name, md5 in table.items() if name == target or name.endswith('/' + target) or name.endswith(target)]
    if len(matches) != 1:
        raise RuntimeError(f'Expected one official CHECKSUM entry for {target}; found {matches}')
    return matches[0][1]

def log_event(event, **fields):
    LOG.parent.mkdir(parents=True, exist_ok=True)
    row = {'time_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'), 'event':event, **fields}
    with LOG.open('a', encoding='utf-8') as f:
        f.write(json.dumps(row, sort_keys=True) + '\n')
        f.flush(); os.fsync(f.fileno())

session = requests.Session()
plan = []
for cap, rel in JOBS:
    checksum_file = CHECKSUM_ROOT / cap / 'CHECKSUM'
    if not checksum_file.is_file():
        raise FileNotFoundError(f'Missing staged official CHECKSUM: {checksum_file}')
    expected_md5 = checksum_for(read_checksums(checksum_file), rel)
    url = f'{BASE}/{cap}/{rel}'
    head = session.head(url, allow_redirects=True, timeout=(20,90))
    head.raise_for_status()
    size = int(head.headers.get('Content-Length','0'))
    if size <= 0:
        raise RuntimeError(f'Could not establish remote size for {cap}/{rel}')
    plan.append({'capture':cap, 'rel':rel, 'url':url, 'size':size, 'md5':expected_md5, 'etag':head.headers.get('ETag','')})

total = sum(x['size'] for x in plan)
print(f'Acquisition plan: {len(plan)} files, {total:,} bytes ({total/1024**3:.2f} GiB).', flush=True)
print(f'Includes one resumed image shard and {len(plan)-1} annotation shards. Approximate transfer at 26.6 MiB/s: {total/(26.6*1024**2)/60:.1f} min, plus Drive hashing.', flush=True)
print(f'All large-file writes go to Drive: {ARCHIVE_ROOT}', flush=True)
print(f'Persistent log: {LOG}', flush=True)

started = time.monotonic()
completed, skipped, failed = [], [], []
for n, item in enumerate(plan, 1):
    if time.monotonic() - started >= MAX_RUN_SECONDS:
        print('Safety time limit reached; remaining files will stay queued for a rerun.', flush=True)
        break
    cap, rel = item['capture'], item['rel']
    final = ARCHIVE_ROOT / cap / rel
    final.parent.mkdir(parents=True, exist_ok=True)
    partial = final.with_name(final.name + '.partial')
    meta = final.with_name(final.name + '.transfer.json')
    if final.is_file():
        h = hashlib.md5()
        with final.open('rb') as f:
            for block in iter(lambda:f.read(CHUNK), b''): h.update(block)
        if final.stat().st_size == item['size'] and h.hexdigest() == item['md5']:
            skipped.append(rel); print(f'[{n}/{len(plan)}] already verified: {rel}', flush=True); continue
        raise RuntimeError(f'Existing final file failed size/official-MD5 check; left untouched: {final}')
    offset = partial.stat().st_size if partial.exists() else 0
    if offset > item['size']:
        raise RuntimeError(f'Partial exceeds remote size; left untouched: {partial}')
    headers = {'Range':f'bytes={offset}-'} if offset else {}
    response = session.get(item['url'], headers=headers, stream=True, allow_redirects=True, timeout=(20,90))
    if offset:
        if response.status_code != 206 or not response.headers.get('Content-Range','').startswith(f'bytes {offset}-'):
            response.close(); raise RuntimeError(f'Server did not honor resume range at {offset} for {rel}; partial left untouched.')
    else:
        response.raise_for_status()
        if response.status_code != 200:
            response.close(); raise RuntimeError(f'Unexpected response {response.status_code} for {rel}')
    md5, sha = hashlib.md5(), hashlib.sha256()
    if offset:
        with partial.open('rb') as f:
            for block in iter(lambda:f.read(CHUNK), b''):
                md5.update(block); sha.update(block)
    mode = 'ab' if offset else 'wb'
    written = offset
    last_report = time.monotonic()
    log_event('shard_start', capture=cap, rel=rel, expected_bytes=item['size'], resume_bytes=offset, etag=item['etag'], official_md5=item['md5'])
    try:
        with response, partial.open(mode) as out:
            for block in response.iter_content(chunk_size=CHUNK):
                if not block: continue
                out.write(block); md5.update(block); sha.update(block); written += len(block)
                now = time.monotonic()
                if now - last_report >= 20:
                    out.flush()
                    print(f'[{n}/{len(plan)}] {rel}: {written/item["size"]:.0%}; {written/1024**3:.2f}/{item["size"]/1024**3:.2f} GiB', flush=True)
                    last_report = now
            out.flush(); os.fsync(out.fileno())
        if written != item['size']:
            raise IOError(f'Length mismatch: got {written}, expected {item["size"]}; resumable partial retained.')
        if md5.hexdigest() != item['md5']:
            raise IOError(f'Official MD5 mismatch for {rel}; partial retained for diagnosis.')
        partial.replace(final)
        metadata = {'status':'complete','capture':cap,'relative_path':rel,'bytes':written,'official_md5':md5.hexdigest(),'sha256':sha.hexdigest(),'etag':item['etag'],'verified_utc':datetime.datetime.now(datetime.timezone.utc).isoformat()}
        tmp = meta.with_name(meta.name + '.tmp')
        tmp.write_text(json.dumps(metadata, indent=2, sort_keys=True), encoding='utf-8'); tmp.replace(meta)
        log_event('shard_complete', capture=cap, rel=rel, bytes=written, md5=md5.hexdigest(), sha256=sha.hexdigest(), etag=item['etag'])
        completed.append(rel)
        print(f'[{n}/{len(plan)}] VERIFIED {rel}: {written:,} bytes; MD5 and SHA-256 saved.', flush=True)
    except Exception as exc:
        log_event('shard_error', capture=cap, rel=rel, bytes_partial=partial.stat().st_size if partial.exists() else 0, error=repr(exc))
        failed.append((rel, repr(exc)))
        print(f'FAILED {rel}: {exc}; partial retained if present.', flush=True)
        if 'Length mismatch' in str(exc) or 'timeout' in str(exc).lower():
            break

print(f'Run finished in {(time.monotonic()-started)/60:.1f} min. verified={len(completed)}, already_verified={len(skipped)}, failed={len(failed)}.', flush=True)
if failed: print('Failures:', failed, flush=True)
print('The general image downloader remains paused; no other captures or shards were scheduled.', flush=True)

In [ ]:
from pathlib import Path
import json
BASE = Path('/content/drive/MyDrive/Persistent Visual-to-MANO Hand Tracker/04_DATASET_STAGING/ReInterHand')
CAPS = [
'm--20210701--1058--0000000--pilot--relightablehandsy--participant0--two-hands',
'm--20220628--1327--BKS383--pilot--ProjectGoliath--ContinuousHandsy--two-hands',
'm--20221007--1215--HIR112--pilot--ProjectGoliathScript--Hands--two-hands',
]
for cap in CAPS:
    root = BASE/'00_archives'/cap
    print('\nCAPTURE', cap)
    for p in sorted(root.rglob('*')):
        if p.is_file() and ('.tar.gz' in p.name or p.name in ('CHECKSUM','cam_params.json','frame_list.txt')):
            print(p.relative_to(root), p.stat().st_size)
    check = BASE/'02_checksums'/cap/'CHECKSUM'
    print('CHECKSUM sample:', check.read_text(errors='replace').splitlines()[:8] if check.exists() else 'MISSING')

In [ ]:
from pathlib import Path
import json, re
BASE=Path('/content/drive/MyDrive/Persistent Visual-to-MANO Hand Tracker/04_DATASET_STAGING/ReInterHand')
CAPS=[
'm--20210701--1058--0000000--pilot--relightablehandsy--participant0--two-hands',
'm--20220628--1327--BKS383--pilot--ProjectGoliath--ContinuousHandsy--two-hands',
'm--20221007--1215--HIR112--pilot--ProjectGoliathScript--Hands--two-hands']
for cap in CAPS:
    print('\n',cap)
    root=BASE/'00_archives'/cap
    for group in ('orig_fits','mano_fits'):
        ps=sorted((root/group).glob('*.tar.gz*'))
        print(group,[(p.name,p.stat().st_size) for p in ps if not p.name.endswith(('.partial','.bak'))])
cap=CAPS[1]
p=BASE/'00_archives'/cap/'Mugsy_cameras'/'envmap_per_segment.tar.gzan'
meta=json.loads(p.with_name(p.name+'.transfer.json').read_text())
print('BKS an archive:',p.stat().st_size,'sidecar:',meta)
print('BKS CHECKSUM expected:',[x for x in (BASE/'02_checksums'/cap/'CHECKSUM').read_text().splitlines() if 'envmap_per_segment.tar.gzan ' in x])

In [ ]:
import hashlib
from pathlib import Path
BASE=Path('/content/drive/MyDrive/Persistent Visual-to-MANO Hand Tracker/04_DATASET_STAGING/ReInterHand')
cap='m--20220628--1327--BKS383--pilot--ProjectGoliath--ContinuousHandsy--two-hands'
p=BASE/'00_archives'/cap/'Mugsy_cameras'/'envmap_per_segment.tar.gzan'
expected='3399222bf2b29a174335d4d913a570d7'
h=hashlib.md5(); n=0
with p.open('rb') as f:
    while True:
        b=f.read(8*1024*1024)
        if not b: break
        h.update(b); n+=len(b)
print('actual bytes:',n,'expected bytes:',2*1024**3)
print('actual MD5:',h.hexdigest(),'official MD5:',expected,'MATCH:',h.hexdigest()==expected)

In [ ]:
# Repair only BKS383's corrupt envmap shard; keep the prior file as a recoverable backup.
import hashlib, json, os, time, datetime
from pathlib import Path
import requests
BASE=Path('/content/drive/MyDrive/Persistent Visual-to-MANO Hand Tracker/04_DATASET_STAGING/ReInterHand')
cap='m--20220628--1327--BKS383--pilot--ProjectGoliath--ContinuousHandsy--two-hands'
rel='Mugsy_cameras/envmap_per_segment.tar.gzan'
target=BASE/'00_archives'/cap/rel
checksum=BASE/'02_checksums'/cap/'CHECKSUM'
expected_md5=next(line.split()[-1].lower() for line in checksum.read_text().splitlines() if line.split() and line.split()[0]==rel)
url='https://fb-baas-f32eacb9-8abb-11eb-b2b8-4857dd089e15.s3.amazonaws.com/ReInterHand/'+cap+'/'+rel
partial=target.with_name(target.name+'.repair.partial')
meta_path=target.with_name(target.name+'.transfer.json')
log=BASE.parent.parent/'_transfer_logs'/'reinterhand_labels_and_resume.jsonl'
log.parent.mkdir(parents=True,exist_ok=True); target.parent.mkdir(parents=True,exist_ok=True)
head=requests.head(url,allow_redirects=True,timeout=(20,90)); head.raise_for_status()
expected_size=int(head.headers['Content-Length'])
assert expected_size==2*1024**3, f'Unexpected source size {expected_size}'
print('Repair source size',expected_size,'official MD5',expected_md5,'existing final size',target.stat().st_size,flush=True)
CHUNK=4*1024*1024

def md5_file(path):
    h=hashlib.md5()
    with path.open('rb') as f:
        for block in iter(lambda:f.read(CHUNK),b''): h.update(block)
    return h.hexdigest()

if partial.exists() and partial.stat().st_size>expected_size:
    partial.rename(partial.with_name(partial.name+f'.oversize-{int(time.time())}.bak'))
offset=partial.stat().st_size if partial.exists() else 0
for attempt in range(1,5):
    headers={'Range':f'bytes={offset}-'} if offset else {}
    with requests.get(url,headers=headers,stream=True,allow_redirects=True,timeout=(20,120)) as response:
        if offset and response.status_code==206:
            cr=response.headers.get('Content-Range','')
            if not cr.startswith(f'bytes {offset}-'):
                raise RuntimeError(f'Bad Content-Range while resuming: {cr!r}')
            mode='ab'
        else:
            if offset and response.status_code==200:
                offset=0
            response.raise_for_status(); mode='wb'
        response.raise_for_status()
        written=offset; last=time.monotonic()
        with partial.open(mode) as out:
            for block in response.iter_content(chunk_size=CHUNK):
                if not block: continue
                out.write(block); written+=len(block)
                if time.monotonic()-last>=20:
                    out.flush(); os.fsync(out.fileno())
                    print(f'Repair transfer: {written/1024**3:.2f}/{expected_size/1024**3:.2f} GiB',flush=True)
                    last=time.monotonic()
            out.flush(); os.fsync(out.fileno())
    if partial.stat().st_size==expected_size:
        break
    offset=partial.stat().st_size
    if attempt==4: raise RuntimeError(f'Repair download incomplete at {offset} bytes')
    print(f'Incomplete response; will resume at {offset} bytes',flush=True)

actual_size=partial.stat().st_size
actual_md5=md5_file(partial)
if actual_size!=expected_size or actual_md5!=expected_md5:
    raise RuntimeError(f'Repair partial failed verification: size={actual_size}, md5={actual_md5}')
print('Repair download verified from Drive partial; promoting it now.',flush=True)
stamp=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
backup=target.with_name(target.name+f'.bad-{target.stat().st_size}-{stamp}.bak')
os.replace(target,backup)
os.replace(partial,target)
final_size=target.stat().st_size
final_md5=md5_file(target)
if final_size!=expected_size or final_md5!=expected_md5:
    failed=target.with_name(target.name+f'.failed-{stamp}.bak')
    os.replace(target,failed)
    os.replace(backup,target)
    raise RuntimeError(f'Promoted file failed verification; restored backup. size={final_size}, md5={final_md5}')
sha=hashlib.sha256()
with target.open('rb') as f:
    for block in iter(lambda:f.read(CHUNK),b''): sha.update(block)
meta={'status':'complete','relative_path':rel,'bytes':final_size,'official_md5':expected_md5,'md5':final_md5,'sha256':sha.hexdigest(),'verified_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),'repair_backup':backup.name}
meta_path.write_text(json.dumps(meta,indent=2,sort_keys=True))
with log.open('a') as f:
    f.write(json.dumps({'time_utc':meta['verified_utc'],'event':'shard_repaired','capture':cap,'relative_path':rel,'bytes':final_size,'md5':final_md5,'sha256':meta['sha256'],'backup':backup.name})+'\n'); f.flush(); os.fsync(f.fileno())
print('FINAL VERIFIED',final_size,'bytes; MD5',final_md5,'SHA-256',meta['sha256'],'backup',backup.name,flush=True)

In [ ]:
# Small extraction probe: the complete BKS383 MANO shard. This uses official paths and writes only to Drive.
from pathlib import Path
import subprocess, json
BASE=Path('/content/drive/MyDrive/Persistent Visual-to-MANO Hand Tracker/04_DATASET_STAGING/ReInterHand')
cap='m--20220628--1327--BKS383--pilot--ProjectGoliath--ContinuousHandsy--two-hands'
source=BASE/'00_archives'/cap/'mano_fits'/'mano_fits.tar.gzaa'
target=BASE/'01_extracted'/cap/'mano_fits'
target.mkdir(parents=True,exist_ok=True)
meta=json.loads(source.with_name(source.name+'.transfer.json').read_text())
assert meta['status']=='complete' and source.stat().st_size==meta['bytes']
print('Extracting verified BKS383 MANO archive to',target,flush=True)
result=subprocess.run(['tar','-xzf',str(source),'-C',str(target),'--no-same-owner','--no-same-permissions'],capture_output=True,text=True)
print('tar exit:',result.returncode,'stderr tail:',result.stderr[-2000:])
files=list((target/'params').glob('*_right.json')) if (target/'params').is_dir() else []
print('right MANO parameter files:',len(files),'sample:',[p.name for p in files[:3]])
if result.returncode: raise RuntimeError('MANO extraction probe failed')
if not files: raise RuntimeError('Probe found no params/*_right.json at expected capture layout')

In [ ]:
# Selective probe: only the files required by records_from_capture (MANO params), not meshes.
from pathlib import Path
import subprocess, time
BASE=Path('/content/drive/MyDrive/Persistent Visual-to-MANO Hand Tracker/04_DATASET_STAGING/ReInterHand')
cap='m--20220628--1327--BKS383--pilot--ProjectGoliath--ContinuousHandsy--two-hands'
source=BASE/'00_archives'/cap/'mano_fits'/'mano_fits.tar.gzaa'
target=BASE/'01_extracted'/cap/'mano_fits'
target.mkdir(parents=True,exist_ok=True)
print('Selective extraction started; input 548 MB; filter=params/*_right.json',flush=True)
t0=time.monotonic()
proc=subprocess.Popen(['tar','--extract','--gzip','--file',str(source),'--directory',str(target),'--no-same-owner','--no-same-permissions','--wildcards','--wildcards-match-slash','params/*_right.json'],stdout=subprocess.DEVNULL,stderr=subprocess.PIPE,text=True)
while proc.poll() is None:
    time.sleep(10)
    print(f'Alive; elapsed={time.monotonic()-t0:.0f}s',flush=True)
err=proc.stderr.read() if proc.stderr else ''
files=list((target/'params').glob('*_right.json')) if (target/'params').is_dir() else []
print('tar exit:',proc.returncode,'elapsed seconds:',round(time.monotonic()-t0,1),'right MANO params:',len(files),'sample:',[p.name for p in files[:3]],'stderr:',err[-1000:])
if proc.returncode or not files: raise RuntimeError('Selective MANO extraction probe failed or matched no files')

In [ ]:
from pathlib import Path
P=Path('/content/drive/MyDrive/Persistent Visual-to-MANO Hand Tracker')
print('Drive project entries:',[p.name for p in P.iterdir()])
for d in [P/'03_EXPERIMENTS',P/'02_CODE',P/'code',P/'hm_ex']:
    if d.exists():
        matches=list(d.rglob('phase5_train_temporal.py'))
        print(d,'phase5 script matches:',[str(x) for x in matches[:5]],'file count top-level:',sum(x.is_file() for x in d.iterdir()))


In [ ]:
from pathlib import Path
P=Path('/content/drive/MyDrive/Persistent Visual-to-MANO Hand Tracker')
S=P/'02_SOURCE_CODE'
print('02_SOURCE_CODE exists:',S.is_dir())
if S.is_dir():
    for p in sorted(S.rglob('*')):
        if p.is_file(): print(p.relative_to(P),p.stat().st_size)


In [ ]:
# Prepare the already-staged ReInterHand captures for training; this cell does not train.
import hashlib, json, os, re, shutil, subprocess, time
from pathlib import Path

PROJECT = Path('/content/drive/MyDrive/Persistent Visual-to-MANO Hand Tracker')
BASE = PROJECT / '04_DATASET_STAGING' / 'ReInterHand'
ARCHIVES = BASE / '00_archives'
CHECKS = BASE / '02_checksums'
EXTRACTED = BASE / '01_extracted'
LOG_DIR = BASE / '03_preparation'
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = LOG_DIR / 'prepare_annotations.jsonl'

CAPTURES = {
    'm--20210701--1058--0000000--pilot--relightablehandsy--participant0--two-hands': {
        'image': ['aa','ab','ac','ad','ae','af','ag','ah','ai','aj','ak','al','am','an','ao','ap','aq','ar','as','at','au','av','aw','ax','ay','az','ba','bb','bc','bd','be','bf','bg','bh','bi','bj','bk','bl'],
        'orig_fits': ['aa','ab','ac','ad'], 'mano_fits': ['aa']},
    'm--20220628--1327--BKS383--pilot--ProjectGoliath--ContinuousHandsy--two-hands': {
        'image': ['aa','ab','ac','ad','ae','af','ag','ah','ai','aj','ak','al','am','an','ao','ap','aq','ar','as','at','au','av'],
        'orig_fits': ['aa','ab'], 'mano_fits': ['aa']},
    'm--20221007--1215--HIR112--pilot--ProjectGoliathScript--Hands--two-hands': {
        'image': ['aa','ab','ac'], 'image_expected_full': ['aa','ab','ac','ad','ae','af','ag','ah','ai','aj','ak','al','am','an','ao','ap','aq','ar','as','at','au','av','aw','ax','ay','az','ba','bb','bc','bd','be','bf','bg','bh','bi','bj','bk','bl','bm','bn','bo','bp','bq','br','bs'],
        'orig_fits': ['aa','ab','ac','ad','ae'], 'mano_fits': ['aa']},
}

def read_md5_table(path):
    table = {}
    for line in path.read_text(encoding='utf-8', errors='replace').splitlines():
        fields = line.strip().split()
        if len(fields) >= 2 and re.fullmatch(r'[0-9a-fA-F]{32}', fields[-1]):
            table[' '.join(fields[:-1]).lstrip('./').replace('\\','/')] = fields[-1].lower()
    return table

def log_event(event, **fields):
    row = {'time_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()), 'event': event, **fields}
    with LOG_PATH.open('a', encoding='utf-8') as f:
        f.write(json.dumps(row, sort_keys=True) + '\n'); f.flush(); os.fsync(f.fileno())

def archive_for(capture, group, suffix):
    if group == 'image': rel = f'Mugsy_cameras/envmap_per_segment.tar.gz{suffix}'
    else: rel = f'{group}/{group}.tar.gz{suffix}'
    return ARCHIVES / capture / rel, rel

def extract_stream(capture, group, suffixes, pattern, target):
    check_file = CHECKS / capture / 'CHECKSUM'
    table = read_md5_table(check_file)
    files = []
    for suffix in suffixes:
        path, rel = archive_for(capture, group, suffix)
        if not path.is_file(): raise FileNotFoundError(f'Missing staged archive: {path}')
        if rel not in table: raise RuntimeError(f'No official CHECKSUM entry for {rel}')
        meta_path = path.with_name(path.name + '.transfer.json')
        meta = json.loads(meta_path.read_text(encoding='utf-8'))
        if meta.get('status') != 'complete' or meta.get('bytes') != path.stat().st_size:
            raise RuntimeError(f'Unverified/incomplete transfer sidecar: {path}')
        files.append((path, rel, table[rel], meta))
    target.mkdir(parents=True, exist_ok=True)
    err_path = LOG_DIR / f'{capture}__{group}.tar.stderr.log'
    cmd = ['tar', '-xzf', '-', '-C', str(target), '--no-same-owner', '--no-same-permissions']
    if pattern:
        cmd += ['--wildcards', '--wildcards-match-slash', pattern]
    with err_path.open('wb') as err:
        proc = subprocess.Popen(cmd, stdin=subprocess.PIPE, stdout=subprocess.DEVNULL, stderr=err, )
        total_bytes = sum(p.stat().st_size for p, *_ in files)
        sent = 0; last = time.monotonic(); started = time.monotonic()
        try:
            for path, rel, expected_md5, meta in files:
                md5 = hashlib.md5(); sha = hashlib.sha256(); this_bytes = 0
                with path.open('rb') as src:
                    while True:
                        block = src.read(8 * 1024 * 1024)
                        if not block: break
                        md5.update(block); sha.update(block); this_bytes += len(block); sent += len(block)
                        proc.stdin.write(block)
                        if time.monotonic() - last >= 30:
                            print(f'{capture} {group}: read {sent/1024**3:.2f}/{total_bytes/1024**3:.2f} GiB; elapsed {time.monotonic()-started:.0f}s', flush=True)
                            last = time.monotonic()
                got_md5, got_sha = md5.hexdigest(), sha.hexdigest()
                if this_bytes != meta['bytes'] or got_md5 != expected_md5:
                    raise RuntimeError(f'Archive integrity failure {rel}: bytes={this_bytes}/{meta["bytes"]}, MD5={got_md5}/{expected_md5}')
                if meta.get('sha256') and got_sha != meta['sha256']:
                    raise RuntimeError(f'Transfer SHA-256 mismatch for {rel}')
                print(f'Verified {rel} ({this_bytes:,} bytes, official MD5 + transfer SHA-256)', flush=True)
                log_event('archive_verified_during_extract', capture=capture, group=group, path=rel, bytes=this_bytes, md5=got_md5, sha256=got_sha)
            proc.stdin.close()
            returncode = proc.wait()
        except BaseException:
            try: proc.stdin.close()
            except Exception: pass
            try: proc.terminate(); proc.wait(timeout=10)
            except Exception: pass
            raise
    elapsed = time.monotonic() - started
    if returncode != 0:
        tail = err_path.read_text(encoding='utf-8', errors='replace')[-1500:]
        raise RuntimeError(f'tar extraction failed for {capture}/{group}, exit={returncode}; stderr tail:\n{tail}')
    log_event('extract_complete', capture=capture, group=group, pattern=pattern or '*', archive_count=len(files), archive_bytes=total_bytes, elapsed_seconds=round(elapsed,1), stderr=str(err_path))
    print(f'Extracted {capture}/{group}: {len(files)} archives, {total_bytes/1024**3:.2f} GiB in {elapsed/60:.1f} min', flush=True)

def setup_metadata(capture):
    root = EXTRACTED / capture
    root.mkdir(parents=True, exist_ok=True)
    src = CHECKS / capture
    for name, dest in [('frame_list.txt', root/'frame_list.txt'), ('frame_list_orig.txt', root/'frame_list_orig.txt'), ('cam_params.json', root/'Mugsy_cameras'/'cam_params.json'), ('CHECKSUM', root/'CHECKSUM')]:
        candidates = [src/name, ARCHIVES/capture/'metadata'/'Mugsy_cameras'/name, src/'metadata'/'Mugsy_cameras'/name]
        found = next((p for p in candidates if p.is_file()), None)
        if found is None:
            if name == 'frame_list_orig.txt': continue
            raise FileNotFoundError(f'Missing required capture metadata {name} under {src}')
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(found, dest)
    return root

print('Preparing capture metadata and right-hand 3D/MANO annotations only; no training.', flush=True)
for capture, plan in CAPTURES.items():
    root = setup_metadata(capture)
    kp_dir = root/'orig_fits'/'right'/'Keypoints'
    mano_dir = root/'mano_fits'/'params'
    print(f'\n{capture}', flush=True)
    if not any(kp_dir.glob('keypoint-*.json')):
        extract_stream(capture, 'orig_fits', plan['orig_fits'], 'orig_fits/right/Keypoints/keypoint-*.json', root)
    else:
        print(f'Reusing existing right keypoints: {sum(1 for _ in kp_dir.glob("keypoint-*.json")):,}', flush=True)
    existing_mano = sum(1 for _ in mano_dir.glob('*_right.json')) if mano_dir.is_dir() else 0
    if capture.endswith('BKS383--pilot--ProjectGoliath--ContinuousHandsy--two-hands') and existing_mano == 14882:
        print('Reusing previously completed selective MANO extraction: 14,882 right parameter JSONs.', flush=True)
        log_event('reuse_verified_mano_probe', capture=capture, right_parameter_files=existing_mano)
    elif existing_mano == 0:
        extract_stream(capture, 'mano_fits', plan['mano_fits'], 'mano_fits/params/*_right.json', root)
    else:
        print(f'Reusing existing MANO params: {existing_mano:,}', flush=True)
    kp_count = sum(1 for _ in kp_dir.glob('keypoint-*.json'))
    mano_count = sum(1 for _ in mano_dir.glob('*_right.json')) if mano_dir.is_dir() else 0
    print(f'Current labels: right keypoint JSONs={kp_count:,}; right MANO parameter JSONs={mano_count:,}', flush=True)
    log_event('annotation_inventory', capture=capture, right_keypoint_files=kp_count, right_mano_parameter_files=mano_count, capture_root=str(root))
print('\nAnnotation extraction stage finished. Image extraction and joined-frame validation remain.', flush=True)


In [ ]:
# Prepare the already-staged ReInterHand captures for training; this cell does not train.
import hashlib, json, os, re, shutil, subprocess, time
from pathlib import Path

PROJECT = Path('/content/drive/MyDrive/Persistent Visual-to-MANO Hand Tracker')
BASE = PROJECT / '04_DATASET_STAGING' / 'ReInterHand'
ARCHIVES = BASE / '00_archives'
CHECKS = BASE / '02_checksums'
EXTRACTED = BASE / '01_extracted'
LOG_DIR = BASE / '03_preparation'
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = LOG_DIR / 'prepare_annotations.jsonl'

CAPTURES = {
    'm--20210701--1058--0000000--pilot--relightablehandsy--participant0--two-hands': {
        'image': ['aa','ab','ac','ad','ae','af','ag','ah','ai','aj','ak','al','am','an','ao','ap','aq','ar','as','at','au','av','aw','ax','ay','az','ba','bb','bc','bd','be','bf','bg','bh','bi','bj','bk','bl'],
        'orig_fits': ['aa','ab','ac','ad'], 'mano_fits': ['aa']},
    'm--20220628--1327--BKS383--pilot--ProjectGoliath--ContinuousHandsy--two-hands': {
        'image': ['aa','ab','ac','ad','ae','af','ag','ah','ai','aj','ak','al','am','an','ao','ap','aq','ar','as','at','au','av'],
        'orig_fits': ['aa','ab'], 'mano_fits': ['aa']},
    'm--20221007--1215--HIR112--pilot--ProjectGoliathScript--Hands--two-hands': {
        'image': ['aa','ab','ac'], 'image_expected_full': ['aa','ab','ac','ad','ae','af','ag','ah','ai','aj','ak','al','am','an','ao','ap','aq','ar','as','at','au','av','aw','ax','ay','az','ba','bb','bc','bd','be','bf','bg','bh','bi','bj','bk','bl','bm','bn','bo','bp','bq','br','bs'],
        'orig_fits': ['aa','ab','ac','ad','ae'], 'mano_fits': ['aa']},
}

def read_md5_table(path):
    table = {}
    for line in path.read_text(encoding='utf-8', errors='replace').splitlines():
        fields = line.strip().split()
        if len(fields) >= 2 and re.fullmatch(r'[0-9a-fA-F]{32}', fields[-1]):
            table[' '.join(fields[:-1]).lstrip('./').replace('\\','/')] = fields[-1].lower()
    return table

def log_event(event, **fields):
    row = {'time_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()), 'event': event, **fields}
    with LOG_PATH.open('a', encoding='utf-8') as f:
        f.write(json.dumps(row, sort_keys=True) + '\n'); f.flush(); os.fsync(f.fileno())

def archive_for(capture, group, suffix):
    if group == 'image': rel = f'Mugsy_cameras/envmap_per_segment.tar.gz{suffix}'
    else: rel = f'{group}/{group}.tar.gz{suffix}'
    return ARCHIVES / capture / rel, rel

def extract_stream(capture, group, suffixes, pattern, target):
    check_file = CHECKS / capture / 'CHECKSUM'
    table = read_md5_table(check_file)
    files = []
    for suffix in suffixes:
        path, rel = archive_for(capture, group, suffix)
        if not path.is_file(): raise FileNotFoundError(f'Missing staged archive: {path}')
        if rel not in table: raise RuntimeError(f'No official CHECKSUM entry for {rel}')
        meta_path = path.with_name(path.name + '.transfer.json')
        meta = json.loads(meta_path.read_text(encoding='utf-8'))
        if meta.get('status') != 'complete' or meta.get('bytes') != path.stat().st_size:
            raise RuntimeError(f'Unverified/incomplete transfer sidecar: {path}')
        files.append((path, rel, table[rel], meta))
    target.mkdir(parents=True, exist_ok=True)
    err_path = LOG_DIR / f'{capture}__{group}.tar.stderr.log'
    cmd = ['tar', '-xzf', '-', '-C', str(target), '--no-same-owner', '--no-same-permissions']
    if pattern:
        cmd += ['--wildcards', '--wildcards-match-slash', pattern]
    with err_path.open('wb') as err:
        proc = subprocess.Popen(cmd, stdin=subprocess.PIPE, stdout=subprocess.DEVNULL, stderr=err, )
        total_bytes = sum(p.stat().st_size for p, *_ in files)
        sent = 0; last = time.monotonic(); started = time.monotonic()
        try:
            for path, rel, expected_md5, meta in files:
                md5 = hashlib.md5(); sha = hashlib.sha256(); this_bytes = 0
                with path.open('rb') as src:
                    while True:
                        block = src.read(8 * 1024 * 1024)
                        if not block: break
                        md5.update(block); sha.update(block); this_bytes += len(block); sent += len(block)
                        proc.stdin.write(block)
                        if time.monotonic() - last >= 30:
                            print(f'{capture} {group}: read {sent/1024**3:.2f}/{total_bytes/1024**3:.2f} GiB; elapsed {time.monotonic()-started:.0f}s', flush=True)
                            last = time.monotonic()
                got_md5, got_sha = md5.hexdigest(), sha.hexdigest()
                if this_bytes != meta['bytes'] or got_md5 != expected_md5:
                    raise RuntimeError(f'Archive integrity failure {rel}: bytes={this_bytes}/{meta["bytes"]}, MD5={got_md5}/{expected_md5}')
                if meta.get('sha256') and got_sha != meta['sha256']:
                    raise RuntimeError(f'Transfer SHA-256 mismatch for {rel}')
                print(f'Verified {rel} ({this_bytes:,} bytes, official MD5 + transfer SHA-256)', flush=True)
                log_event('archive_verified_during_extract', capture=capture, group=group, path=rel, bytes=this_bytes, md5=got_md5, sha256=got_sha)
            proc.stdin.close()
            returncode = proc.wait()
        except BaseException:
            try: proc.stdin.close()
            except Exception: pass
            try: proc.terminate(); proc.wait(timeout=10)
            except Exception: pass
            raise
    elapsed = time.monotonic() - started
    if returncode != 0:
        tail = err_path.read_text(encoding='utf-8', errors='replace')[-1500:]
        raise RuntimeError(f'tar extraction failed for {capture}/{group}, exit={returncode}; stderr tail:\n{tail}')
    log_event('extract_complete', capture=capture, group=group, pattern=pattern or '*', archive_count=len(files), archive_bytes=total_bytes, elapsed_seconds=round(elapsed,1), stderr=str(err_path))
    print(f'Extracted {capture}/{group}: {len(files)} archives, {total_bytes/1024**3:.2f} GiB in {elapsed/60:.1f} min', flush=True)

def setup_metadata(capture):
    root = EXTRACTED / capture
    root.mkdir(parents=True, exist_ok=True)
    src = CHECKS / capture
    for name, dest in [('frame_list.txt', root/'frame_list.txt'), ('frame_list_orig.txt', root/'frame_list_orig.txt'), ('cam_params.json', root/'Mugsy_cameras'/'cam_params.json'), ('CHECKSUM', root/'CHECKSUM')]:
        candidates = [src/name, ARCHIVES/capture/'metadata'/'Mugsy_cameras'/name, src/'metadata'/'Mugsy_cameras'/name]
        found = next((p for p in candidates if p.is_file()), None)
        if found is None:
            if name == 'frame_list_orig.txt': continue
            raise FileNotFoundError(f'Missing required capture metadata {name} under {src}')
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(found, dest)
    return root

print('Preparing capture metadata and right-hand 3D/MANO annotations only; no training.', flush=True)
for capture, plan in CAPTURES.items():
    root = setup_metadata(capture)
    kp_dir = root/'orig_fits'/'right'/'Keypoints'
    mano_dir = root/'mano_fits'/'params'
    print(f'\n{capture}', flush=True)
    if not any(kp_dir.glob('keypoint-*.json')):
        extract_stream(capture, 'orig_fits', plan['orig_fits'], 'right/Keypoints/keypoint-*.json', root/'orig_fits')
    else:
        print(f'Reusing existing right keypoints: {sum(1 for _ in kp_dir.glob("keypoint-*.json")):,}', flush=True)
    existing_mano = sum(1 for _ in mano_dir.glob('*_right.json')) if mano_dir.is_dir() else 0
    if capture.endswith('BKS383--pilot--ProjectGoliath--ContinuousHandsy--two-hands') and existing_mano == 14882:
        print('Reusing previously completed selective MANO extraction: 14,882 right parameter JSONs.', flush=True)
        log_event('reuse_verified_mano_probe', capture=capture, right_parameter_files=existing_mano)
    elif existing_mano == 0:
        extract_stream(capture, 'mano_fits', plan['mano_fits'], 'params/*_right.json', root/'mano_fits')
    else:
        print(f'Reusing existing MANO params: {existing_mano:,}', flush=True)
    kp_count = sum(1 for _ in kp_dir.glob('keypoint-*.json'))
    mano_count = sum(1 for _ in mano_dir.glob('*_right.json')) if mano_dir.is_dir() else 0
    print(f'Current labels: right keypoint JSONs={kp_count:,}; right MANO parameter JSONs={mano_count:,}', flush=True)
    log_event('annotation_inventory', capture=capture, right_keypoint_files=kp_count, right_mano_parameter_files=mano_count, capture_root=str(root))
print('\nAnnotation extraction stage finished. Image extraction and joined-frame validation remain.', flush=True)


In [ ]:
from pathlib import Path
import json, time, shutil
P=Path('/content/drive/MyDrive/Persistent Visual-to-MANO Hand Tracker/04_DATASET_STAGING/ReInterHand')
CAPS={
 '202107':'m--20210701--1058--0000000--pilot--relightablehandsy--participant0--two-hands',
 'BKS383':'m--20220628--1327--BKS383--pilot--ProjectGoliath--ContinuousHandsy--two-hands',
 'HIR112':'m--20221007--1215--HIR112--pilot--ProjectGoliathScript--Hands--two-hands'}
print('Drive mounted:',Path('/content/drive/MyDrive').exists(),'runtime free GiB:',round(shutil.disk_usage('/content').free/1024**3,1),flush=True)
for label,cap in CAPS.items():
 r=P/'01_extracted'/cap
 kp=r/'orig_fits/right/Keypoints'; mano=r/'mano_fits/params'
 print(label,'root=',r.exists(),'keypoints=',sum(1 for _ in kp.glob('keypoint-*.json')) if kp.exists() else 0,'mano=',sum(1 for _ in mano.glob('*_right.json')) if mano.exists() else 0,'frame_list=',(r/'frame_list.txt').is_file(),flush=True)
cap=CAPS['HIR112']; root=P/'00_archives'/cap/'orig_fits'
print('HIR orig archives:',[(x.name,x.stat().st_size) for x in sorted(root.glob('orig_fits.tar.gz*')) if x.is_file() and '.transfer.json' not in x.name],flush=True)
q=sorted(root.glob('orig_fits.tar.gzaa'))[0]; t=time.monotonic()
with q.open('rb') as f: sample=f.read(4*1024*1024)
print('4MiB Drive read:',len(sample),'bytes in',round(time.monotonic()-t,2),'s',flush=True)

In [ ]:
from pathlib import Path
import time, shutil
P=Path('/content/drive/MyDrive/Persistent Visual-to-MANO Hand Tracker/04_DATASET_STAGING/ReInterHand')
cap='m--20221007--1215--HIR112--pilot--ProjectGoliathScript--Hands--two-hands'
checks=[('drive',Path('/content/drive/MyDrive')),('archives',P/'00_archives'/cap),('hir orig aa',P/'00_archives'/cap/'orig_fits'/'orig_fits.tar.gzaa'),('hir mano aa',P/'00_archives'/cap/'mano_fits'/'mano_fits.tar.gzaa'),('hir extracted root',P/'01_extracted'/cap)]
print('probe start', flush=True)
for name,path in checks:
 t=time.monotonic()
 try: print(name, 'exists=',path.exists(), 'is_file=',path.is_file(), 'bytes=',path.stat().st_size if path.is_file() else '-', 'seconds=',round(time.monotonic()-t,2),flush=True)
 except Exception as e: print(name,repr(e),flush=True)
print('runtime free GiB',round(shutil.disk_usage('/content').free/1024**3,1),flush=True)

In [ ]:
from pathlib import Path
P=Path('/content/drive/MyDrive/Persistent Visual-to-MANO Hand Tracker/02_SOURCE_CODE/Data_Pipeline')
for fn in ['phase6_extract_interhand_capture.py','reinterhand_sequences.py']:
 p=P/fn; print('\n###',fn,'bytes',p.stat().st_size,flush=True)
 lines=p.read_text(encoding='utf-8').splitlines()
 for i,line in enumerate(lines):
  if any(k in line.lower() for k in ['tarfile','tar.gz','image_path','image_dir','jpeg','jpg','png','frame_list','manifest','extract']):
   print(f'{i+1}: {line[:220]}')

In [ ]:
from pathlib import Path
P=Path('/content/drive/MyDrive/Persistent Visual-to-MANO Hand Tracker/02_SOURCE_CODE/Data_Pipeline')
for fn,a,b in [('phase6_extract_interhand_capture.py',80,210),('reinterhand_sequences.py',150,235)]:
 lines=(P/fn).read_text(encoding='utf-8').splitlines(); print('\n###',fn,flush=True)
 for i in range(a-1,min(b,len(lines))): print(f'{i+1}: {lines[i]}')

In [ ]:
from pathlib import Path
import time, signal
P=Path('/content/drive/MyDrive/Persistent Visual-to-MANO Hand Tracker/04_DATASET_STAGING/ReInterHand/00_archives/m--20221007--1215--HIR112--pilot--ProjectGoliathScript--Hands--two-hands/orig_fits/orig_fits.tar.gzaa')
def timeout(*_): raise TimeoutError('Drive read exceeded 45 seconds')
signal.signal(signal.SIGALRM, timeout); signal.alarm(45)
t=time.monotonic()
try:
 with P.open('rb') as f: data=f.read(4*1024*1024)
 print('read bytes',len(data),'seconds',round(time.monotonic()-t,2),flush=True)
finally: signal.alarm(0)

In [ ]:
from pathlib import Path
P=Path('/content/drive/MyDrive/Persistent Visual-to-MANO Hand Tracker/02_SOURCE_CODE/Data_Pipeline/phase6_make_reinterhand_anchor.py')
L=P.read_text(encoding='utf-8').splitlines()
for i,line in enumerate(L):
 if any(k in line.lower() for k in ['def ','manifest','frame_list','records_from','capture_root','image_path','keypoint','mano','write_text','csv.writer','output']): print(f'{i+1}: {line[:220]}')

In [ ]:
from pathlib import Path
P=Path('/content/drive/MyDrive/Persistent Visual-to-MANO Hand Tracker/02_SOURCE_CODE/Data_Pipeline/phase6_extract_interhand_capture.py')
L=P.read_text(encoding='utf-8').splitlines()
for a,b in [(1,80)]:
 for i in range(a-1,min(b,len(L))): print(f'{i+1}: {L[i]}')

In [ ]:
from pathlib import Path
import json, os, signal, time
BASE = Path('/content/drive/MyDrive/Persistent Visual-to-MANO Hand Tracker/04_DATASET_STAGING/ReInterHand')
CAPS = {'202107':'m--20210701--1058--0000000--pilot--relightablehandsy--participant0--two-hands','BKS383':'m--20220628--1327--BKS383--pilot--ProjectGoliath--ContinuousHandsy--two-hands','HIR112':'m--20221007--1215--HIR112--pilot--ProjectGoliathScript--Hands--two-hands'}
def bounded_count(folder, ending, seconds=30):
    if not folder.is_dir(): return 'missing'
    start=time.monotonic(); count=0
    def timeout(*_): raise TimeoutError('directory scan hit time bound')
    old=signal.signal(signal.SIGALRM, timeout); signal.alarm(seconds)
    try:
        with os.scandir(folder) as entries:
            for e in entries:
                if e.is_file() and e.name.endswith(ending): count += 1
    except TimeoutError:
        return 'timeout after %ds; counted at least %d' % (seconds,count)
    finally:
        signal.alarm(0); signal.signal(signal.SIGALRM,old)
    return '%d files in %.1fs' % (count,time.monotonic()-start)
for label,cap in CAPS.items():
    checks=BASE/'02_checksums'/cap
    extracted=BASE/'01_extracted'/cap
    print('CAPTURE',label,flush=True)
    for rel in ['Mugsy_cameras/cam_params.json','frame_list.txt','frame_list_orig.txt']:
        a=checks/rel; b=extracted/rel
        print(rel,'metadata=',a.is_file(),'extracted=',b.is_file(),flush=True)
    print('keypoint count:',bounded_count(extracted/'orig_fits'/'right'/'Keypoints','.json'),flush=True)
    print('MANO count:',bounded_count(extracted/'mano_fits'/'params','_right.json'),flush=True)


In [ ]:
from google.colab import drive
from pathlib import Path
import shutil
if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive')
PROJECT=Path('/content/drive/MyDrive/Persistent Visual-to-MANO Hand Tracker')
print('Drive MyDrive:',Path('/content/drive/MyDrive').exists())
print('Project:',PROJECT.exists(),PROJECT)
print('staging:',(PROJECT/'04_DATASET_STAGING/ReInterHand').exists())
print('runtime free GiB:',round(shutil.disk_usage('/content').free/1024**3,1))


In [ ]:
# Read-only inventory of already-acquired ReInterHand data; no downloads or extraction.
from pathlib import Path
import json, re, os, time, shutil
BASE = Path('/content/drive/MyDrive/Persistent Visual-to-MANO Hand Tracker/04_DATASET_STAGING/ReInterHand')
CAPS = {
 '202107': ('m--20210701--1058--0000000--pilot--relightablehandsy--participant0--two-hands', 38),
 'BKS383': ('m--20220628--1327--BKS383--pilot--ProjectGoliath--ContinuousHandsy--two-hands', 22),
 'HIR112': ('m--20221007--1215--HIR112--pilot--ProjectGoliathScript--Hands--two-hands', 45),
}
print('Drive mounted:', Path('/content/drive/MyDrive').is_dir(), '| free runtime GiB:', round(shutil.disk_usage('/content').free/2**30,1), flush=True)
for label,(cap,expected_shards) in CAPS.items():
    ex=BASE/'01_extracted'/cap; ar=BASE/'00_archives'/cap; meta=BASE/'02_checksums'/cap
    print('\n###',label, '| extracted root:',ex.exists(),flush=True)
    fl=ex/'frame_list.txt'; frames=[]; segs=set()
    if fl.is_file():
        for line in fl.open(encoding='utf-8',errors='replace'):
            p=line.split()
            if len(p)>=2:
                try: frames.append(int(p[1])); segs.add(p[0])
                except ValueError: pass
        print('frame_list rows:',len(frames),'unique frame IDs:',len(set(frames)),'segments:',len(segs),'duplicates:',len(frames)-len(set(frames)),flush=True)
    else: print('frame_list MISSING',flush=True)
    cam=ex/'Mugsy_cameras/cam_params.json'
    if cam.is_file():
        try:
            d=json.loads(cam.read_text()); print('camera metadata:',len(d),'entries; sample IDs:',list(d)[:5],flush=True)
        except Exception as e: print('camera JSON INVALID:',repr(e),flush=True)
    else: print('camera metadata MISSING',flush=True)
    for name,folder,pat in [('right keypoints','orig_fits/right/Keypoints','keypoint-*.json'),('right MANO','mano_fits/params','*_right.json')]:
        d=ex/folder; ids=set(); n=0
        if d.is_dir():
            with os.scandir(d) as it:
                for ent in it:
                    if ent.is_file() and (ent.name.startswith('keypoint-') and ent.name.endswith('.json') if name=='right keypoints' else ent.name.endswith('_right.json')):
                        n+=1; m=re.search(r'(\d+)',ent.name)
                        if m: ids.add(int(m.group(1)))
        print(name+':',n,'| overlap frame_list:',len(ids.intersection(frames)),'| frame IDs unmatched:',len(set(frames)-ids),'| extras:',len(ids-set(frames)),flush=True)
        if n:
            sample=next((d/p.name for p in d.iterdir() if p.is_file() and (p.name.endswith('.json'))),None)
            if sample:
                try:
                    obj=json.loads(sample.read_text()); print(' sample JSON',sample.name,'type',type(obj).__name__,'keys',list(obj)[:8] if isinstance(obj,dict) else 'len='+str(len(obj)),flush=True)
                except Exception as e: print(' sample JSON INVALID',repr(e),flush=True)
    image_root=ex/'Mugsy_cameras/envmap_per_segment/images'; img_n=0; camdirs=[]
    if image_root.is_dir():
        for dn,dirs,files in os.walk(image_root):
            camdirs.append((Path(dn).relative_to(image_root).as_posix(),len(files)))
            img_n+=sum(Path(f).suffix.lower() in ('.png','.jpg','.jpeg','.webp') for f in files)
            if len(camdirs)>=1000: break
    print('extracted image files:',img_n,'camera dirs (first):',camdirs[:8],flush=True)
    stem='envmap_per_segment.tar.gz'; finals=[]; parts=[]
    if ar.exists():
        for p in ar.rglob(stem+'*'):
            if p.is_file() and '.transfer.json' not in p.name:
                suf=p.name[len(stem):]
                if suf.endswith('.partial'): parts.append((suf,p.stat().st_size))
                elif len(suf)==2 and suf.isalpha() and suf.islower(): finals.append((suf,p.stat().st_size))
    print('image archive shards:',len(finals),'of official',expected_shards,'| archived GiB:',round(sum(b for _,b in finals)/2**30,2),'| suffixes:',','.join(s for s,_ in sorted(finals)),'| partials:',parts,flush=True)
print('\nInventory complete; read-only.')

In [ ]:
# Inspect only the first 40 tar headers from one already-complete image archive stream.
from pathlib import Path
import tarfile, re, time
BASE=Path('/content/drive/MyDrive/Persistent Visual-to-MANO Hand Tracker/04_DATASET_STAGING/ReInterHand/00_archives')
CAP='m--20210701--1058--0000000--pilot--relightablehandsy--participant0--two-hands'
paths=sorted((BASE/CAP/'Mugsy_cameras').glob('envmap_per_segment.tar.gz??'))
assert len(paths)==38, f'expected 38 complete 202107 parts, found {len(paths)}'
class Joined:
    def __init__(self, paths): self.paths=list(paths); self.i=0; self.f=self.paths[0].open('rb') if self.paths else None
    def read(self,n=-1):
        if n is None or n<0: raise ValueError('stream reader requires bounded reads')
        out=bytearray()
        while n and self.f:
            b=self.f.read(n)
            if b: out.extend(b); n-=len(b)
            else:
                self.f.close(); self.i+=1
                self.f=self.paths[self.i].open('rb') if self.i<len(self.paths) else None
        return bytes(out)
    def close(self):
        if self.f: self.f.close(); self.f=None
    def readable(self): return True
r=Joined(paths); t=time.monotonic(); rows=[]
try:
    with tarfile.open(fileobj=r,mode='r|gz') as tf:
        for i,m in enumerate(tf):
            rows.append((m.name,m.size,m.type.decode('ascii','replace') if isinstance(m.type,bytes) else str(m.type)))
            if i>=39: break
finally: r.close()
print('parts:',len(paths),'header probe seconds:',round(time.monotonic()-t,1))
for row in rows: print(row)
print('probe done; archive read-only.')

In [ ]:
from pathlib import Path
import tarfile, time
AR=Path('/content/drive/MyDrive/Persistent Visual-to-MANO Hand Tracker/04_DATASET_STAGING/ReInterHand/00_archives')
CAP='m--20221007--1215--HIR112--pilot--ProjectGoliathScript--Hands--two-hands'
parts=sorted((AR/CAP/'mano_fits').glob('mano_fits.tar.gz??'))
r=Joined(parts); t=time.monotonic(); out=[]
try:
    with tarfile.open(fileobj=r,mode='r|gz') as tf:
        for i,m in enumerate(tf):
            out.append((m.name,m.size,m.isfile()))
            if i>=24: break
finally: r.close()
print('HIR MANO archive probe:',len(parts),'part(s),',round(time.monotonic()-t,1),'sec')
for x in out: print(x)# Resumable, checksum-logged extraction of the already-acquired HIR112 MANO archive.
# Reads only Drive archives; writes extracted members and an audit manifest to Drive.
from pathlib import Path, PurePosixPath
import hashlib, json, os, tarfile, time
BASE=Path('/content/drive/MyDrive/Persistent Visual-to-MANO Hand Tracker/04_DATASET_STAGING/ReInterHand')
CAP='m--20221007--1215--HIR112--pilot--ProjectGoliathScript--Hands--two-hands'
parts=sorted((BASE/'00_archives'/CAP/'mano_fits').glob('mano_fits.tar.gz??'))
assert len(parts)==1, f'expected HIR112 MANO archive part, found {len(parts)}'
for part in parts:
    side=part.with_name(part.name+'.transfer.json')
    if not side.is_file(): raise RuntimeError(f'missing transfer sidecar: {side.name}')
    meta=json.loads(side.read_text())
    if meta.get('status')!='complete' or meta.get('bytes')!=part.stat().st_size or not meta.get('sha256'):
        raise RuntimeError(f'archive is not marked complete/size-verified: {part.name}')
OUT=BASE/'01_extracted'/CAP/'mano_fits'
OUT.mkdir(parents=True,exist_ok=True)
LOG=BASE/'03_preparation'/f'{CAP}_mano_extract.jsonl'
LOG.parent.mkdir(parents=True,exist_ok=True)
known={}
if LOG.is_file():
    with LOG.open(encoding='utf-8') as f:
        for line in f:
            try:
                e=json.loads(line); known[e['member']]=(int(e['size']),e['sha256'])
            except Exception: pass
class PartStream:
    def __init__(self, paths): self.paths=list(paths); self.i=0; self.f=self.paths[0].open('rb') if self.paths else None
    def read(self,n=-1):
        if n is None or n<0: raise ValueError('bounded stream read required')
        out=bytearray()
        while n and self.f:
            b=self.f.read(n)
            if b: out.extend(b); n-=len(b)
            else:
                self.f.close(); self.i+=1
                self.f=self.paths[self.i].open('rb') if self.i<len(self.paths) else None
        return bytes(out)
    def close(self):
        if self.f: self.f.close(); self.f=None
    def readable(self): return True
stream=PartStream(parts); started=time.monotonic(); last=time.monotonic(); seen=written=skipped=0; written_bytes=0; pending=[]
def flush_log():
    if pending:
        with LOG.open('a',encoding='utf-8') as f:
            f.writelines(json.dumps(x,separators=(',',':'))+'\n' for x in pending); f.flush(); os.fsync(f.fileno())
        pending.clear()
print('Extract HIR112 MANO meshes + params from',parts[0].name,'to Drive; archive bytes',parts[0].stat().st_size,flush=True)
try:
    with tarfile.open(fileobj=stream,mode='r|gz') as tf:
        for member in tf:
            if not member.isfile(): continue
            rel=PurePosixPath(member.name)
            if rel.is_absolute() or '..' in rel.parts or not rel.parts: raise RuntimeError(f'unsafe archive path: {member.name!r}')
            dest=(OUT/Path(*rel.parts)).resolve()
            if OUT.resolve() not in dest.parents: raise RuntimeError(f'archive path escaped output: {member.name!r}')
            dest.parent.mkdir(parents=True,exist_ok=True)
            prior=known.get(member.name)
            can_skip=bool(prior and prior[0]==member.size and dest.is_file() and dest.stat().st_size==member.size)
            tmp=dest.with_name(dest.name+'.extracting')
            source=tf.extractfile(member)
            if source is None: raise RuntimeError(f'cannot read tar member: {member.name}')
            h=hashlib.sha256(); remaining=member.size
            with source:
                if can_skip:
                    while remaining:
                        block=source.read(min(4*1024*1024,remaining))
                        if not block: raise EOFError(f'truncated member {member.name}')
                        h.update(block); remaining-=len(block)
                else:
                    with tmp.open('wb') as out:
                        while remaining:
                            block=source.read(min(4*1024*1024,remaining))
                            if not block: raise EOFError(f'truncated member {member.name}')
                            out.write(block); h.update(block); remaining-=len(block)
                        out.flush(); os.fsync(out.fileno())
            digest=h.hexdigest(); seen+=1
            if can_skip:
                if digest!=prior[1]: raise RuntimeError(f'archive member digest changed since prior extraction: {member.name}')
                skipped+=1
            else:
                if tmp.stat().st_size!=member.size: raise IOError(f'wrong extracted size for {member.name}')
                os.replace(tmp,dest); written+=1; written_bytes+=member.size
                pending.append({'member':member.name,'size':member.size,'sha256':digest})
            if len(pending)>=128 or time.monotonic()-last>=30:
                flush_log(); last=time.monotonic()
                print(f'HIR112 MANO: members={seen:,}; written={written:,}; reused={skipped:,}; new={written_bytes/2**30:.2f} GiB; elapsed={(time.monotonic()-started)/60:.1f} min',flush=True)
    flush_log()
except Exception as exc:
    flush_log(); stream.close(); raise
finally:
    stream.close()
print(f'HIR112 MANO extraction pass finished: members={seen:,}, newly written={written:,}, reused={skipped:,}, new bytes={written_bytes:,}, elapsed={(time.monotonic()-started)/60:.1f} min',flush=True)


In [ ]:
# Streaming read-only QC for acquired labels and archive coverage; no downloads or extraction.
from pathlib import Path
import os, json, time, shutil
BASE = Path('/content/drive/MyDrive/Persistent Visual-to-MANO Hand Tracker/04_DATASET_STAGING/ReInterHand')
CAPS = [
    ('202107', 'm--20210701--1058--0000000--pilot--relightablehandsy--participant0--two-hands'),
    ('BKS383', 'm--20220628--1327--BKS383--pilot--ProjectGoliath--ContinuousHandsy--two-hands'),
    ('HIR112', 'm--20221007--1215--HIR112--pilot--ProjectGoliathScript--Hands--two-hands'),
]
def scan_ids(folder, prefix, suffix, label):
    ids, files, sample = set(), 0, None
    if not folder.is_dir(): return ids, files, sample
    with os.scandir(folder) as entries:
        for entry in entries:
            if not entry.is_file(follow_symlinks=False): continue
            name=entry.name
            if not (name.startswith(prefix) and name.endswith(suffix)): continue
            token=name[len(prefix):-len(suffix)] if suffix else name[len(prefix):]
            if not token.isdigit(): continue
            files += 1
            if sample is None: sample=folder/name
            ids.add(int(token))
            if label == 'HIR112 MANO' and files % 25000 == 0:
                print(f'  scanned {files:,} HIR112 MANO files', flush=True)
    return ids, files, sample
print('STREAMING DATASET QC — existing Drive files only', flush=True)
for label, cap in CAPS:
    started=time.monotonic()
    root=BASE/'01_extracted'/cap
    print(f'\n[{label}] root_exists={root.is_dir()}', flush=True)
    frame_file=root/'frame_list.txt'
    frames,segments=[],set()
    if frame_file.is_file():
        with frame_file.open(encoding='utf-8',errors='replace') as stream:
            for line in stream:
                fields=line.split()
                if len(fields)>=2:
                    try: frames.append(int(fields[1])); segments.add(fields[0])
                    except ValueError: pass
    frame_set=set(frames)
    cam_file=root/'Mugsy_cameras'/'cam_params.json'
    try: camera_count=len(json.loads(cam_file.read_text(encoding='utf-8'))) if cam_file.is_file() else 0
    except Exception: camera_count=-1
    kp_ids,kp_count,kp_sample=scan_ids(root/'orig_fits'/'right'/'Keypoints','keypoint-','.json',label+' keypoints')
    mano_ids,mano_count,mano_sample=scan_ids(root/'mano_fits'/'params','', '_right.json',label+' MANO')
    print(f'  frame_list={len(frames):,} rows/{len(frame_set):,} unique IDs; segments={len(segments)}; cameras={camera_count}',flush=True)
    print(f'  right keypoints={kp_count:,}; frame IDs missing labels={len(frame_set-kp_ids):,}; extra IDs={len(kp_ids-frame_set):,}',flush=True)
    print(f'  right MANO params={mano_count:,}; frame IDs missing fits={len(frame_set-mano_ids):,}; extra IDs={len(mano_ids-frame_set):,}; keypoint/MANO ID mismatch={len(kp_ids ^ mano_ids):,}',flush=True)
    for kind,sample in [('keypoint',kp_sample),('MANO',mano_sample)]:
        if sample is None: continue
        try:
            obj=json.loads(sample.read_text(encoding='utf-8'))
            if kind=='keypoint':
                import numpy as np
                arr=np.asarray(obj,dtype=np.float64)
                print(f'  sample {kind}: JSON valid; shape={arr.shape}; finite={bool(np.isfinite(arr).all())}',flush=True)
            else: print(f'  sample {kind}: JSON valid; keys={list(obj)[:8] if isinstance(obj,dict) else type(obj).__name__}',flush=True)
        except Exception as exc: print(f'  sample {kind}: INVALID ({type(exc).__name__}: {exc})',flush=True)
    image_dir=root/'Mugsy_cameras'/'envmap_per_segment'/'images'/'400485'
    if image_dir.is_dir():
        n=sum(1 for e in os.scandir(image_dir) if e.is_file(follow_symlinks=False) and e.name.lower().endswith(('.png','.jpg','.jpeg')))
        print(f'  extracted images camera 400485={n:,}',flush=True)
    else: print('  extracted images camera 400485=0 (directory absent)',flush=True)
    archive_root=BASE/'00_archives'/cap
    for group,pattern in [('image','envmap_per_segment.tar.gz??'),('orig_fits','orig_fits.tar.gz??'),('mano_fits','mano_fits.tar.gz??')]:
        folder=archive_root/('Mugsy_cameras' if group=='image' else group)
        paths=[p for p in folder.glob(pattern) if p.is_file() and not p.name.endswith('.partial')] if folder.is_dir() else []
        print(f'  {group} archives={len(paths)}; bytes={sum(p.stat().st_size for p in paths):,}',flush=True)
    print(f'  QC scan elapsed={(time.monotonic()-started):.1f}s',flush=True)
usage=shutil.disk_usage('/content')
print(f'\nQC finished; local runtime disk free={usage.free/2**30:.1f} GiB (persistent outputs remain on Drive).',flush=True)

In [ ]:
from pathlib import Path
import time
P=Path('/content/drive/MyDrive/Persistent Visual-to-MANO Hand Tracker/04_DATASET_STAGING/ReInterHand')
checks=[
 ('Drive mount',Path('/content/drive/MyDrive')),
 ('202107 frame_list',P/'01_extracted/m--20210701--1058--0000000--pilot--relightablehandsy--participant0--two-hands/frame_list.txt'),
 ('202107 camera JSON',P/'01_extracted/m--20210701--1058--0000000--pilot--relightablehandsy--participant0--two-hands/Mugsy_cameras/cam_params.json'),
 ('202107 image shard aa',P/'00_archives/m--20210701--1058--0000000--pilot--relightablehandsy--participant0--two-hands/Mugsy_cameras/envmap_per_segment.tar.gzaa'),
 ('202107 MANO params dir',P/'01_extracted/m--20210701--1058--0000000--pilot--relightablehandsy--participant0--two-hands/mano_fits/params'),
]
for label,path in checks:
 t=time.monotonic()
 try:
  exists=path.exists(); isfile=path.is_file()
  size=path.stat().st_size if isfile else None
  prefix=path.open('rb').read(48).hex() if isfile and size and size < 1000000 else None
  print(f'{label}: exists={exists}, file={isfile}, bytes={size}, read_prefix={prefix}, elapsed={time.monotonic()-t:.2f}s',flush=True)
 except Exception as e: print(f'{label}: ERROR {type(e).__name__}: {e}',flush=True)

In [ ]:
# Resumable extraction of image archives already present on Drive.
# Does not download shards, train models, or delete/overwrite unrelated files.
import io, os, re, json, time, tarfile, shutil
from pathlib import Path, PurePosixPath
import cv2
import numpy as np

PROJECT = Path('/content/drive/MyDrive/Persistent Visual-to-MANO Hand Tracker')
STAGING = PROJECT / '04_DATASET_STAGING' / 'ReInterHand'
ARCHIVES = STAGING / '00_archives'
EXTRACTED = STAGING / '01_extracted'
LOG_DIR = STAGING / '_extract_logs'
LOG_DIR.mkdir(parents=True, exist_ok=True)
CAPTURES = [
    ('202107', 'm--20210701--1058--0000000--pilot--relightablehandsy--participant0--two-hands', 38),
    ('BKS383', 'm--20220628--1327--BKS383--pilot--ProjectGoliath--ContinuousHandsy--two-hands', 22),
    ('HIR112', 'm--20221007--1215--HIR112--pilot--ProjectGoliathScript--Hands--two-hands', 45),
]
STEM = 'envmap_per_segment.tar.gz'

def suffix_key(p):
    return p.name[len(STEM):]

def log_line(path, obj):
    with path.open('a', encoding='utf-8') as f:
        f.write(json.dumps(obj, sort_keys=True) + '\n')
        f.flush()

def copy_small_metadata(cap):
    src = STAGING / '02_checksums' / cap
    dst = EXTRACTED / cap
    (dst / 'Mugsy_cameras').mkdir(parents=True, exist_ok=True)
    copied = []
    for rel in ('frame_list.txt', 'frame_list_orig.txt', 'Mugsy_cameras/cam_params.json'):
        a, b = src / rel, dst / rel
        if a.is_file() and not b.is_file():
            b.parent.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(a, b)
            copied.append(rel)
    return copied

class PartStream(io.RawIOBase):
    def __init__(self, paths):
        super().__init__()
        self.paths = list(paths)
        self.i = 0
        self.fp = None
    def readable(self):
        return True
    def readinto(self, b):
        view = memoryview(b)
        done = 0
        while done < len(view):
            if self.fp is None:
                if self.i >= len(self.paths):
                    break
                self.fp = self.paths[self.i].open('rb')
                self.i += 1
            n = self.fp.readinto(view[done:])
            if not n:
                self.fp.close()
                self.fp = None
                continue
            done += n
        return done
    def close(self):
        if self.fp is not None:
            self.fp.close()
            self.fp = None
        super().close()

def verified_parts(cap, expected_n):
    folder = ARCHIVES / cap / 'Mugsy_cameras'
    found = sorted(folder.glob(STEM + '??'), key=suffix_key)
    parts = []
    for p in found:
        suf = suffix_key(p)
        if not re.fullmatch(r'[a-z]{2}', suf):
            continue
        side = p.with_name(p.name + '.transfer.json')
        if not side.is_file():
            raise RuntimeError(f'Missing transfer sidecar: {side}')
        meta = json.loads(side.read_text(encoding='utf-8'))
        if meta.get('status') != 'complete' or int(meta.get('bytes', -1)) != p.stat().st_size:
            raise RuntimeError(f'Incomplete/mismatched image shard: {p.name}')
        parts.append(p)
    if not parts:
        raise FileNotFoundError(f'No finalized image shards for {cap}')
    suffixes = [suffix_key(p) for p in parts]
    if len(parts) != expected_n:
        print(f'  SOURCE PREFIX ONLY: {len(parts)}/{expected_n} official shards present; extraction will stop at this prefix.', flush=True)
    # Refuse gaps: a concatenated tar stream cannot safely skip a missing part.
    expected = []
    for n in range(expected_n):
        a, b = divmod(n, 26)
        expected.append(chr(97 + a) + chr(97 + b))
    for i, suf in enumerate(suffixes):
        if suf != expected[i]:
            raise RuntimeError(f'Non-contiguous archive sequence for {cap}: expected {expected[i]}, found {suf}')
    return parts

def extract_capture(label, cap, expected_n):
    started = time.monotonic()
    parts = verified_parts(cap, expected_n)
    root = EXTRACTED / cap
    root.mkdir(parents=True, exist_ok=True)
    metadata_copied = copy_small_metadata(cap)
    manifest = LOG_DIR / f'{label}_images.jsonl'
    errors = LOG_DIR / f'{label}_bad_or_unreadable.jsonl'
    summary_path = LOG_DIR / f'{label}_images_summary.json'
    seen = {}
    if manifest.is_file():
        with manifest.open(encoding='utf-8') as f:
            for line in f:
                try:
                    row = json.loads(line)
                    seen[row['member']] = row
                except Exception:
                    continue
    print(f'[{label}] reading {len(parts)}/{expected_n} sequential shards ({sum(p.stat().st_size for p in parts)/1024**3:.2f} GiB compressed); prior manifest entries={len(seen):,}', flush=True)
    reader = io.BufferedReader(PartStream(parts), buffer_size=1024*1024)
    archive = None
    count = skipped = bad = decoded_bytes = 0
    cameras, frame_ids = set(), set()
    last_report = time.monotonic()
    manifest_fp = manifest.open('a', encoding='utf-8')
    error_fp = errors.open('a', encoding='utf-8')
    stream_error = None
    try:
        archive = tarfile.open(fileobj=reader, mode='r|gz')
        for member in archive:
            if not member.isfile():
                continue
            name = member.name.replace('\\', '/')
            posix = PurePosixPath(name)
            if posix.is_absolute() or '..' in posix.parts:
                bad += 1
                error_fp.write(json.dumps({'member': name, 'error': 'unsafe archive path'}) + '\n')
                continue
            if not name.startswith('envmap_per_segment/images/'):
                continue
            ext = posix.suffix.lower()
            if ext not in ('.png', '.jpg', '.jpeg'):
                continue
            rel = str(posix)
            dest = root / 'Mugsy_cameras' / Path(*posix.parts)
            prev = seen.get(rel)
            if prev and dest.is_file() and dest.stat().st_size == int(prev.get('bytes', -1)):
                skipped += 1
            else:
                src = archive.extractfile(member)
                if src is None:
                    bad += 1
                    error_fp.write(json.dumps({'member': name, 'error': 'tar member has no payload'}) + '\n')
                    continue
                blob = src.read()
                image = cv2.imdecode(np.frombuffer(blob, dtype=np.uint8), cv2.IMREAD_UNCHANGED)
                if image is None:
                    bad += 1
                    error_fp.write(json.dumps({'member': name, 'error': 'image decode failed', 'bytes': len(blob)}) + '\n')
                    continue
                h, w = image.shape[:2]
                dest.parent.mkdir(parents=True, exist_ok=True)
                tmp = dest.with_name(dest.name + '.extracting')
                with tmp.open('wb') as f:
                    f.write(blob)
                    f.flush()
                os.replace(tmp, dest)
                row = {'member': rel, 'bytes': len(blob), 'width': int(w), 'height': int(h)}
                manifest_fp.write(json.dumps(row, separators=(',', ':')) + '\n')
                count += 1
                decoded_bytes += len(blob)
                if len(posix.parts) >= 4:
                    cameras.add(posix.parts[-2])
                    if posix.stem.isdigit():
                        frame_ids.add(posix.stem)
            if (count + skipped) % 512 == 0:
                manifest_fp.flush(); error_fp.flush()
            now = time.monotonic()
            if now - last_report >= 30:
                manifest_fp.flush(); error_fp.flush()
                print(f'  [{label}] new={count:,} existing={skipped:,} bad={bad:,}; images={count+skipped:,}; {decoded_bytes/1024**3:.2f} GiB new; {now-started:.1f} s', flush=True)
                last_report = now
    except (EOFError, OSError, tarfile.TarError, ValueError) as exc:
        stream_error = f'{type(exc).__name__}: {exc}'
        print(f'  [{label}] stream ended with {stream_error}', flush=True)
    finally:
        manifest_fp.flush(); error_fp.flush()
        manifest_fp.close(); error_fp.close()
        try:
            if archive is not None: archive.close()
        except Exception:
            pass
        reader.close()
    complete_source = len(parts) == expected_n
    summary = {
        'capture': cap, 'label': label, 'available_shards': len(parts),
        'expected_shards': expected_n, 'source_complete': complete_source,
        'new_images_this_run': count, 'existing_manifest_images_skipped': skipped,
        'bad_images_or_members': bad, 'decoded_bytes_this_run': decoded_bytes,
        'camera_ids_seen_this_run': sorted(cameras), 'frame_ids_seen_this_run': len(frame_ids),
        'metadata_copied': metadata_copied, 'stream_error': stream_error,
        'elapsed_seconds': round(time.monotonic()-started, 1),
        'manifest': str(manifest), 'errors': str(errors),
    }
    tmp_summary = summary_path.with_suffix('.json.tmp')
    tmp_summary.write_text(json.dumps(summary, indent=2), encoding='utf-8')
    os.replace(tmp_summary, summary_path)
    print(f'[{label}] pass complete: new={count:,}, skipped={skipped:,}, decode/path issues={bad:,}, available shards={len(parts)}/{expected_n}, elapsed={(time.monotonic()-started)/60:.1f} min', flush=True)
    return summary

if not (Path('/content/drive/MyDrive').is_dir() and PROJECT.is_dir()):
    raise RuntimeError('Expected mounted Google Drive project is unavailable.')
results = []
for label, cap, n in CAPTURES:
    results.append(extract_capture(label, cap, n))
print('\nEXTRACTION SUMMARY')
for r in results:
    print(f"{r['label']}: {r['available_shards']}/{r['expected_shards']} shards; new={r['new_images_this_run']:,}; skipped={r['existing_manifest_images_skipped']:,}; bad={r['bad_images_or_members']:,}; source_complete={r['source_complete']}; elapsed={r['elapsed_seconds']/60:.1f} min")
print(f'Persistent logs: {LOG_DIR}')


In [ ]:
# Resume image extraction from Drive shards already present. No network access, downloads, or training.
from pathlib import Path
import json, time, os

_required = ("ARCHIVES", "EXTRACTED", "LOG_DIR", "STEM", "extract_capture", "CAPTURES")
_missing = [name for name in _required if name not in globals()]
if _missing:
    raise RuntimeError("The failed extraction cell's kernel state is unavailable: " + ", ".join(_missing) + ". Do not rerun the full notebook; reconnect the existing runtime first.")

# Return only a contiguous prefix whose existing transfer sidecars match the files.
# At the first absent/corrupt shard, stop; later shards cannot be concatenated across a gap.
PREFIX_AUDIT = {}
def verified_parts(cap, expected_n):
    folder = ARCHIVES / cap / "Mugsy_cameras"
    parts = []
    stop_reason = None
    for n in range(expected_n):
        suffix = chr(97 + n // 26) + chr(97 + n % 26)
        path = folder / (STEM + suffix)
        side = path.with_name(path.name + ".transfer.json")
        if not path.is_file():
            stop_reason = f"missing shard {suffix}"
            break
        try:
            size = path.stat().st_size
            meta = json.loads(side.read_text(encoding="utf-8"))
        except Exception as exc:
            stop_reason = f"cannot read shard/sidecar {suffix}: {type(exc).__name__}: {exc}"
            break
        if meta.get("status") != "complete":
            stop_reason = f"shard {suffix} sidecar status={meta.get('status')!r}"
            break
        if int(meta.get("bytes", -1)) != size:
            stop_reason = f"shard {suffix} size mismatch: file={size}, sidecar={meta.get('bytes')}"
            break
        if not meta.get("sha256"):
            stop_reason = f"shard {suffix} has no SHA-256 sidecar"
            break
        parts.append(path)
    if not parts:
        raise RuntimeError(f"No verified contiguous image shards for {cap}: {stop_reason}")
    PREFIX_AUDIT[cap] = {
        "label": next((label for label, c, _ in CAPTURES if c == cap), cap),
        "verified_prefix": [p.name[-2:] for p in parts],
        "verified_shards": len(parts), "expected_shards": expected_n,
        "prefix_bytes": sum(p.stat().st_size for p in parts),
        "source_complete": len(parts) == expected_n,
        "first_unusable_shard_reason": stop_reason,
    }
    print(f"[{PREFIX_AUDIT[cap]['label']}] verified existing prefix: {len(parts)}/{expected_n} shards; "
          f"{PREFIX_AUDIT[cap]['prefix_bytes']/1024**3:.2f} GiB; stop: {stop_reason or 'complete'}", flush=True)
    return parts

# Preflight both targets before writing anything. The completed 202107 capture is skipped.
recovery_targets = [(label, cap, n) for label, cap, n in CAPTURES if label in ("BKS383", "HIR112")]
for label, cap, n in recovery_targets:
    target_images = EXTRACTED / cap / "Mugsy_cameras" / "envmap_per_segment" / "images"
    manifest = LOG_DIR / f"{label}_images.jsonl"
    if target_images.exists() and not manifest.is_file():
        raise RuntimeError(f"Refusing to overwrite unmanifested existing image directory: {target_images}")
    verified_parts(cap, n)

_audit_path = LOG_DIR / "image_source_prefix_recovery.json"
def _save_audit():
    tmp = _audit_path.with_suffix(".json.tmp")
    tmp.write_text(json.dumps({"mode": "existing_drive_shards_only", "captures": PREFIX_AUDIT}, indent=2), encoding="utf-8")
    os.replace(tmp, _audit_path)
_save_audit()

recovery_results = []
for label, cap, n in recovery_targets:
    print(f"\n[{label}] extracting from existing verified prefix only", flush=True)
    summary = extract_capture(label, cap, n)
    recovery_results.append(summary)
    PREFIX_AUDIT[cap]["extraction_summary"] = summary
    _save_audit()
    print(f"[{label}] source_complete={summary['source_complete']}; new={summary['new_images_this_run']:,}; "
          f"resumed/skipped={summary['existing_manifest_images_skipped']:,}; "
          f"bad={summary['bad_images_or_members']:,}; stream_end={summary['stream_error']}", flush=True)

print("\nRECOVERY EXTRACTION SUMMARY (existing Drive data only)")
for r in recovery_results:
    print(f"{r['label']}: {r['available_shards']}/{r['expected_shards']} shards; "
          f"new={r['new_images_this_run']:,}; skipped={r['existing_manifest_images_skipped']:,}; "
          f"decode/path issues={r['bad_images_or_members']:,}; source_complete={r['source_complete']}; "
          f"stream_end={r['stream_error']}")
print(f"Prefix audit: {_audit_path}")


In [ ]:
# Read-only QA of currently extracted ReInterHand data. No downloads, extraction, or training.
from pathlib import Path
import os, re, json, math, time
import numpy as np
import cv2

BASE = Path('/content/drive/MyDrive/Persistent Visual-to-MANO Hand Tracker/04_DATASET_STAGING/ReInterHand')
CAPTURES = [
    ('202107', 'm--20210701--1058--0000000--pilot--relightablehandsy--participant0--two-hands'),
    ('BKS383', 'm--20220628--1327--BKS383--pilot--ProjectGoliath--ContinuousHandsy--two-hands'),
    ('HIR112', 'm--20221007--1215--HIR112--pilot--ProjectGoliathScript--Hands--two-hands'),
]

# Reuse the per-shard prefix audit produced by the prior extraction cell.
prefix_path = BASE / '_extract_logs' / 'image_source_prefix_recovery.json'
prefix_audit = None
try:
    if prefix_path.is_file():
        prefix_audit = json.loads(prefix_path.read_text(encoding='utf-8'))
except Exception as exc:
    print('Prefix audit read issue:', type(exc).__name__, str(exc)[:200], flush=True)


def scan_ids(folder, pattern, label):
    """Read names once; retry transient Google Drive FUSE listing failures."""
    rx = re.compile(pattern)
    last_error = None
    for attempt in range(1, 4):
        names = {}
        collisions = 0
        t0 = time.monotonic()
        try:
            with os.scandir(folder) as entries:
                for n, entry in enumerate(entries, 1):
                    match = rx.fullmatch(entry.name)
                    if not match:
                        continue
                    key = int(match.group(1))
                    if key in names and names[key] != entry.name:
                        collisions += 1
                    else:
                        names[key] = entry.name
                    if n % 25000 == 0:
                        print(f'  {label}: listed {n:,} names in {time.monotonic()-t0:.1f}s', flush=True)
            return names, collisions, None
        except OSError as exc:
            last_error = f'{type(exc).__name__}: {exc}'
            print(f'  {label}: Drive listing attempt {attempt}/3 failed: {last_error}', flush=True)
            if attempt < 3:
                time.sleep(attempt * 2)
    return None, None, last_error


def finite_numeric_count(obj):
    count = 0
    finite = True
    def visit(x):
        nonlocal count, finite
        if isinstance(x, dict):
            for v in x.values(): visit(v)
        elif isinstance(x, (list, tuple)):
            for v in x: visit(v)
        elif isinstance(x, (int, float)) and not isinstance(x, bool):
            count += 1
            finite = finite and math.isfinite(float(x))
    visit(obj)
    return count, finite


def sample_images(root, limit=3):
    candidates = [
        root / 'Mugsy_cameras' / 'envmap_per_segment',
        root / 'Mugsy_cameras',
    ]
    image_root = next((p for p in candidates if p.is_dir()), None)
    if image_root is None:
        return {'root': None, 'samples': [], 'error': 'image directory not found'}
    found, errors = [], []
    def onerror(exc):
        errors.append(f'{type(exc).__name__}: {exc}')
    for walk_root, dirs, files in os.walk(image_root, onerror=onerror):
        dirs.sort()
        for name in files:
            if not name.lower().endswith(('.png', '.jpg', '.jpeg')):
                continue
            path = Path(walk_root) / name
            image = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
            if image is None or image.size == 0:
                found.append({'path': str(path.relative_to(root)), 'ok': False, 'shape': None})
            else:
                found.append({'path': str(path.relative_to(root)), 'ok': True, 'shape': list(image.shape), 'dtype': str(image.dtype)})
            if len(found) >= limit:
                return {'root': str(image_root.relative_to(root)), 'samples': found, 'walk_errors': errors[:3]}
    return {'root': str(image_root.relative_to(root)), 'samples': found, 'walk_errors': errors[:3]}


report = {
    'scope': 'existing Drive data only; no downloads, extraction, or training',
    'prefix_audit_path': str(prefix_path),
    'prefix_audit': prefix_audit,
    'captures': [],
}
print('READ-ONLY DATASET QA — current extracted files', flush=True)
if prefix_audit is not None:
    print('Image prefix audit loaded:', json.dumps(prefix_audit, sort_keys=True)[:1200], flush=True)

for label, cap in CAPTURES:
    t0 = time.monotonic()
    root = BASE / '01_extracted' / cap
    item = {'label': label, 'capture': cap, 'root_exists': root.is_dir()}
    print()
    print(f'[{label}] root_exists={root.is_dir()}', flush=True)
    frame_path = root / 'frame_list.txt'
    frames, segments, duplicate_rows = [], set(), 0
    if frame_path.is_file():
        seen_rows = set()
        with frame_path.open('r', encoding='utf-8', errors='replace') as stream:
            for line in stream:
                fields = line.split()
                if len(fields) < 2:
                    continue
                try:
                    frame_id = int(fields[1])
                except ValueError:
                    continue
                row = (fields[0], frame_id)
                duplicate_rows += int(row in seen_rows)
                seen_rows.add(row)
                frames.append(frame_id)
                segments.add(fields[0])
    frame_ids = set(frames)
    item.update({'frame_list_exists': frame_path.is_file(), 'frame_rows': len(frames), 'unique_frame_ids': len(frame_ids), 'segments': len(segments), 'duplicate_segment_frame_rows': duplicate_rows})

    cam_path = root / 'Mugsy_cameras' / 'cam_params.json'
    camera_result = {'exists': cam_path.is_file(), 'camera_count': 0, 'valid_intrinsics': False, 'error': None}
    if cam_path.is_file():
        try:
            camera_doc = json.loads(cam_path.read_text(encoding='utf-8'))
            camera_result['camera_count'] = len(camera_doc)
            checks = []
            for cam in camera_doc.values():
                R = np.asarray(cam['R'], dtype=np.float64)
                t = np.asarray(cam['t'], dtype=np.float64)
                focal = np.asarray(cam['focal'], dtype=np.float64)
                principal = np.asarray(cam['princpt'], dtype=np.float64)
                checks.append(R.shape == (3, 3) and t.size == 3 and focal.size == 2 and principal.size == 2 and np.isfinite(R).all() and np.isfinite(t).all() and np.isfinite(focal).all() and np.isfinite(principal).all() and (focal > 0).all())
            camera_result['valid_intrinsics'] = bool(checks) and all(checks)
        except Exception as exc:
            camera_result['error'] = f'{type(exc).__name__}: {exc}'
    item['camera'] = camera_result

    kp_dir = root / 'orig_fits' / 'right' / 'Keypoints'
    mano_dir = root / 'mano_fits' / 'params'
    kp_names, kp_collisions, kp_error = scan_ids(kp_dir, r'keypoint-(\d+)\.json', f'{label} keypoints') if kp_dir.is_dir() else (None, None, 'directory missing')
    mano_names, mano_collisions, mano_error = scan_ids(mano_dir, r'(\d+)_right\.json', f'{label} MANO') if mano_dir.is_dir() else (None, None, 'directory missing')
    label_info = {}
    if kp_names is not None:
        kp_ids = set(kp_names)
        label_info['keypoints'] = {'files': len(kp_names), 'frame_ids_missing': len(frame_ids - kp_ids), 'extra_frame_ids': len(kp_ids - frame_ids), 'filename_id_collisions': kp_collisions}
    else:
        kp_ids = set()
        label_info['keypoints'] = {'scan_error': kp_error}
    if mano_names is not None:
        mano_ids = set(mano_names)
        label_info['mano_right'] = {'files': len(mano_names), 'frame_ids_missing': len(frame_ids - mano_ids), 'extra_frame_ids': len(mano_ids - frame_ids), 'filename_id_collisions': mano_collisions}
    else:
        mano_ids = set()
        label_info['mano_right'] = {'scan_error': mano_error}
    shared = sorted(frame_ids & kp_ids & mano_ids)
    label_info['joint_frame_ids'] = len(shared)
    label_info['joint_frame_ids_missing_of_frame_list'] = len(frame_ids - set(shared))
    label_info['sample_content'] = []
    for frame_id in shared[:5]:
        sample = {'frame_id': frame_id}
        try:
            kp_obj = json.loads((kp_dir / kp_names[frame_id]).read_text(encoding='utf-8'))
            kp_arr = np.asarray(kp_obj, dtype=np.float64)
            sample['keypoints'] = {'shape': list(kp_arr.shape), 'finite': bool(np.isfinite(kp_arr).all()), 'expected_24x3': kp_arr.shape == (24, 3)}
        except Exception as exc:
            sample['keypoints'] = {'error': f'{type(exc).__name__}: {exc}'}
        try:
            mano_obj = json.loads((mano_dir / mano_names[frame_id]).read_text(encoding='utf-8'))
            n_numeric, finite = finite_numeric_count(mano_obj)
            sample['mano'] = {'top_level_keys': sorted(mano_obj)[:20] if isinstance(mano_obj, dict) else [], 'numeric_values': n_numeric, 'finite': finite}
        except Exception as exc:
            sample['mano'] = {'error': f'{type(exc).__name__}: {exc}'}
        label_info['sample_content'].append(sample)
    item['labels'] = label_info
    item['image_samples'] = sample_images(root)
    item['elapsed_seconds'] = round(time.monotonic() - t0, 1)
    report['captures'].append(item)
    print(f"  frame_list={item['frame_rows']:,} rows / {item['unique_frame_ids']:,} unique IDs; segments={item['segments']}; duplicates={duplicate_rows}", flush=True)
    print(f"  cameras={camera_result['camera_count']} valid_intrinsics={camera_result['valid_intrinsics']}; image_samples={len(item['image_samples']['samples'])}", flush=True)
    print('  labels:', json.dumps(label_info, sort_keys=True), flush=True)
    print(f"  image sample results: {json.dumps(item['image_samples'], sort_keys=True)[:1000]}", flush=True)
    print(f"  elapsed={item['elapsed_seconds']:.1f}s", flush=True)

out = BASE / '_extract_logs' / 'qualitative_dataset_qc.json'
out.parent.mkdir(parents=True, exist_ok=True)
tmp = out.with_suffix('.json.tmp')
tmp.write_text(json.dumps(report, indent=2, sort_keys=True), encoding='utf-8')
tmp.replace(out)
print()
print(f'Audit report saved: {out}', flush=True)


In [ ]:
# ONE-CELL REINTERHAND REPAIR + FULL ACQUIRED-IMAGE AUDIT (NO TRAINING)
# Safe to rerun after a disconnect. Only three already-acquired captures are in scope.
import os, io, re, csv, json, time, gzip, shutil, hashlib, tarfile, threading, traceback
from pathlib import Path, PurePosixPath
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
import requests
import numpy as np
import cv2
from google.colab import drive

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Persistent Visual-to-MANO Hand Tracker/04_DATASET_STAGING/ReInterHand')
assert (ROOT / '00_archives').is_dir(), 'Existing dataset folder is unavailable; check Drive mount.'
ARC, EXT = ROOT/'00_archives', ROOT/'01_extracted'
OUT = ROOT/'03_training_ready'
OUT.mkdir(exist_ok=True)
WORK = Path('/content/reinterhand_repair_work'); WORK.mkdir(exist_ok=True)
BASE = 'https://fb-baas-f32eacb9-8abb-11eb-b2b8-4857dd089e15.s3.amazonaws.com/ReInterHand'
CAPS = [
 ('BKS383','m--20220628--1327--BKS383--pilot--ProjectGoliath--ContinuousHandsy--two-hands',22),
 ('HIR112','m--20221007--1215--HIR112--pilot--ProjectGoliathScript--Hands--two-hands',3),
 ('202107','m--20210701--1058--0000000--pilot--relightablehandsy--participant0--two-hands',38),
]
STEM='envmap_per_segment.tar.gz'
GIB=1024**3
MAX_DOWNLOAD=8*GIB # replacement/repair only; never acquire the remaining HIR capture
NEW_BYTES=0
START=time.monotonic()
STATUS={'phase':'starting','images':0}
HALT=threading.Event()
cv2.setNumThreads(1)

def say(s): print(time.strftime('%H:%M:%S'), s, flush=True)
def heartbeat():
    while not HALT.wait(60):
        say('HEARTBEAT '+json.dumps(STATUS)+'; elapsed=%.1f min; free local=%.1f GiB'%((time.monotonic()-START)/60,shutil.disk_usage('/content').free/GIB))
threading.Thread(target=heartbeat,daemon=True).start()

def retry(fn, attempts=5):
    for a in range(attempts):
        try: return fn()
        except (OSError, requests.RequestException) as e:
            if a==attempts-1: raise
            say('I/O retry %d/%d: %s'%(a+1,attempts,str(e)[:150])); time.sleep(min(20,2**a))

def digest(p, kind='sha256'):
    h=hashlib.new(kind)
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(8*1024**2),b''): h.update(b)
    return h.hexdigest()

def atomic_bytes(path,data):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_name(path.name+'.repair_tmp')
    def put():
        with open(tmp,'wb') as f: f.write(data); f.flush(); os.fsync(f.fileno())
        os.replace(tmp,path)
    retry(put)
    if retry(path.read_bytes)!=data: raise IOError('Readback mismatch: '+str(path))

def save_json(path,obj): atomic_bytes(path,json.dumps(obj,indent=2,sort_keys=True).encode())

def persist_file(local,dest):
    expected=digest(local)
    dest=Path(dest); dest.parent.mkdir(parents=True,exist_ok=True)
    tmp=dest.with_name(dest.name+'.repair_tmp')
    retry(lambda:shutil.copyfile(local,tmp))
    if retry(lambda:digest(tmp))!=expected: raise IOError('Drive copy checksum mismatch: '+str(dest))
    retry(lambda:os.replace(tmp,dest))
    return expected

# Save precisely this executable cell for recovery, alongside its outputs.
try: atomic_bytes(OUT/'repair_existing_data.py',get_ipython().history_manager.input_hist_raw[-1].encode())
except NameError: pass

SESSION=requests.Session()
SESSION.headers.update({'Accept-Encoding':'identity'})
def download(url,dest,expected_md5=None,byte_range=None):
    global NEW_BYTES
    dest=Path(dest)
    for attempt in range(4):
        headers={}
        offset=dest.stat().st_size if dest.exists() and byte_range is None else 0
        if byte_range is not None: headers['Range']='bytes=%d-%d'%byte_range
        elif offset: headers['Range']='bytes=%d-'%offset
        try:
            with SESSION.get(url,headers=headers,stream=True,timeout=(30,90)) as r:
                r.raise_for_status()
                if byte_range is not None:
                    if r.status_code!=206 or not r.headers.get('Content-Range','').startswith('bytes %d-'%byte_range[0]):
                        raise RuntimeError('Server did not honor the bounded repair range')
                    mode='wb'
                elif offset and r.status_code==206:
                    if not r.headers.get('Content-Range','').startswith('bytes %d-'%offset): raise RuntimeError('Wrong resume range')
                    mode='ab'
                else: mode='wb'
                with open(dest,mode) as f:
                    for b in r.iter_content(4*1024**2):
                        if not b: continue
                        if NEW_BYTES+len(b)>MAX_DOWNLOAD: raise RuntimeError('Repair download budget exceeded; no bulk acquisition permitted')
                        f.write(b); NEW_BYTES+=len(b)
                    f.flush(); os.fsync(f.fileno())
            if byte_range is not None and dest.stat().st_size!=byte_range[1]-byte_range[0]+1:
                raise IOError('Short bounded range')
            if expected_md5 and digest(dest,'md5')!=expected_md5:
                dest.unlink(missing_ok=True) # only the private local staging copy
                raise IOError('Source MD5 mismatch; retry full replacement')
            return
        except (OSError,requests.RequestException) as e:
            if attempt==3: raise
            say('Download retry: '+str(e)[:150]); time.sleep(3*(attempt+1))

CHECKS={}; RECEIPTS={}
def metadata(cap):
    dest=OUT/cap/'metadata'; dest.mkdir(parents=True,exist_ok=True)
    for rel in ['CHECKSUM','frame_list.txt','Mugsy_cameras/cam_params.json']:
        local=WORK/(hashlib.sha1((cap+rel).encode()).hexdigest()+'.meta')
        download(BASE+'/'+cap+'/'+rel,local)
        atomic_bytes(dest/rel,local.read_bytes())
    checks={}
    for line in (dest/'CHECKSUM').read_text().splitlines():
        fields=line.split()
        md=[x for x in fields if re.fullmatch('[0-9a-fA-F]{32}',x)]
        if len(md)==1:
            names=[x.lstrip('*./') for x in fields if x!=md[0]]
            if names: checks[names[0]]=md[0].lower()
    if not checks: raise RuntimeError('Cannot parse official CHECKSUM')
    CHECKS[cap]=checks
    frames={}
    for line in (dest/'frame_list.txt').read_text().splitlines():
        if not line.strip(): continue
        seg,fid=line.split()[:2]; fid=int(fid)
        if fid in frames and frames[fid]!=seg: raise RuntimeError('Ambiguous temporal frame ID')
        frames[fid]=seg
    cams=json.loads((dest/'Mugsy_cameras/cam_params.json').read_text())
    for key,c in cams.items():
        R=np.asarray(c['R'],float); t=np.asarray(c['t'],float).reshape(3)
        assert R.shape==(3,3) and np.isfinite(R).all() and np.isfinite(t).all()
        assert np.allclose(R@R.T,np.eye(3),atol=.02) and np.linalg.det(R)>.95
        assert np.asarray(c['focal']).shape==(2,) and min(c['focal'])>0
        assert np.isfinite(c['princpt']).all()
    return frames,cams

# Stage only one part at a time (usually 2 GiB), verify official MD5, then read.
# A repaired archive part is preserved on Drive; a bad original is renamed, never deleted.
def stage_part(cap,rel):
    expected=CHECKS[cap].get(rel)
    if expected is None: raise RuntimeError('No official checksum for '+rel)
    local=WORK/'current_archive_part.bin'
    source=ARC/cap/rel
    STATUS.update(phase='checksum/stage '+cap.split('--')[3]+' '+Path(rel).name)
    if source.exists(): retry(lambda:shutil.copyfile(source,local))
    else: local.unlink(missing_ok=True)
    if not local.exists() or digest(local,'md5')!=expected:
        say('Repairing '+rel)
        # The short source may be resumed, but only an exact official MD5 is accepted.
        download(BASE+'/'+cap+'/'+rel,local,expected)
        tmp=source.with_name(source.name+'.verified_repair')
        persist_file(local,tmp)
        if source.exists():
            backup=source.with_name(source.name+'.before_repair_'+time.strftime('%Y%m%d_%H%M%S'))
            retry(lambda:os.replace(source,backup))
        retry(lambda:os.replace(tmp,source))
    got=digest(local,'md5')
    if got!=expected: raise RuntimeError('Checksum verification failed')
    RECEIPTS[cap+'/'+rel]={'bytes':local.stat().st_size,'md5':got,'official_md5_match':True}
    save_json(OUT/'archive_receipts.json',RECEIPTS)
    return local

class Parts(io.RawIOBase):
    def __init__(self,cap,rels,extension=None):
        self.cap,self.rels,self.extension=cap,list(rels),extension
        self.n=0; self.fp=None; self.used_extension=False
    def readable(self): return True
    def readinto(self,b):
        while True:
            if self.fp is None:
                if self.n<len(self.rels): p=stage_part(self.cap,self.rels[self.n]); self.n+=1
                elif self.extension and not self.used_extension:
                    p=self.extension; self.used_extension=True
                else: return 0
                self.fp=open(p,'rb')
            n=self.fp.readinto(b)
            if n: return n
            self.fp.close(); self.fp=None
    def close(self):
        if self.fp: self.fp.close()
        super().close()

def archive_rels(cap,folder,stem,count=None):
    if count is not None:
        suffixes=[chr(97+i//26)+chr(97+i%26) for i in range(count)]
        return [folder+'/'+stem+s for s in suffixes]
    rels=sorted(k for k in CHECKS[cap] if re.fullmatch(re.escape(folder+'/'+stem)+'[a-z]{2}',k))
    if not rels: raise RuntimeError('Missing annotation archive definitions: '+stem)
    return rels

def tar_open(cap,rels,extension=None):
    raw=Parts(cap,rels,extension)
    return raw,tarfile.open(fileobj=io.BufferedReader(raw,4*1024**2),mode='r|gz',bufsize=1024**2)

def labels(cap,frames):
    out=OUT/cap; out.mkdir(parents=True,exist_ok=True)
    # Validated compact arrays replace slow random access to hundreds of thousands of JSON files.
    ids=np.array(sorted(frames),dtype=np.int64); lookup={int(v):i for i,v in enumerate(ids)}
    pose=np.full((len(ids),2,48),np.nan,np.float32)
    shape=np.full((len(ids),2,10),np.nan,np.float32)
    trans=np.full((len(ids),2,3),np.nan,np.float32)
    kp=np.full((len(ids),2,24,3),np.nan,np.float32)
    bad=[]
    for kind,folder,stem in [('mano','mano_fits','mano_fits.tar.gz'),('kp','orig_fits','orig_fits.tar.gz')]:
        STATUS.update(phase='annotation audit '+kind)
        raw,tar=tar_open(cap,archive_rels(cap,folder,stem))
        try:
            for member in tar:
                if not member.isfile() or not member.name.endswith('.json'): continue
                name=member.name.replace('\\','/')
                if kind=='mano':
                    m=re.search(r'(\d+)_(left|right)\.json$',name)
                    if not m: continue
                    fid,hand=int(m[1]),m[2]
                else:
                    m=re.search(r'(?:^|/)(left|right)/Keypoints/keypoint-(\d+)\.json$',name)
                    if not m: continue
                    hand,fid=m[1],int(m[2])
                if fid not in lookup: continue
                i,h=lookup[fid],['left','right'].index(hand)
                obj=json.load(tar.extractfile(member))
                try:
                    if kind=='mano':
                        vals=[np.asarray(obj[k],np.float32).reshape(-1) for k in ('pose','shape','trans')]
                        assert [len(v) for v in vals]==[48,10,3] and all(np.isfinite(v).all() for v in vals)
                        pose[i,h],shape[i,h],trans[i,h]=vals
                    else:
                        a=np.asarray(obj,np.float32); assert a.shape==(24,3) and np.isfinite(a).all()
                        assert np.max(np.linalg.norm(a-a[:1],axis=-1))>1
                        kp[i,h]=a
                except (AssertionError,ValueError,TypeError) as e:
                    bad.append({'frame':fid,'hand':hand,'source':name,'error':type(e).__name__})
        finally: tar.close(); raw.close()
    mano_valid=np.isfinite(pose).all(-1)&np.isfinite(shape).all(-1)&np.isfinite(trans).all(-1)
    kp_valid=np.isfinite(kp).all((-1,-2))
    local=WORK/'labels.npz'
    np.savez_compressed(local,frame_ids=ids,mano_pose_axis_angle=pose,mano_shape=shape,mano_trans_source=trans,
                        keypoints_world_mm=kp,mano_valid=mano_valid,keypoints_valid=kp_valid,hand_order=np.array(['left','right']))
    sha=persist_file(local,out/'labels.npz')
    save_json(out/'label_audit.json',{'frames':len(ids),'mano_valid_left_right':mano_valid.sum(0).tolist(),
              'keypoints_valid_left_right':kp_valid.sum(0).tolist(),'malformed':bad,'labels_sha256':sha})
    say('Labels verified: %d frames; MANO %s; keypoints %s'%(len(ids),mano_valid.sum(0),kp_valid.sum(0)))
    return ids,lookup,mano_valid,kp_valid,kp

def valid_path(name):
    pp=PurePosixPath(name)
    if pp.is_absolute() or '..' in pp.parts: raise RuntimeError('Unsafe archive member')
    parts=list(pp.parts)
    if 'envmap_per_segment' not in parts: return None
    return PurePosixPath('Mugsy_cameras',*parts[parts.index('envmap_per_segment'):]).as_posix()

def verify_write_image(job):
    cap,rel,data=job
    decoded=cv2.imdecode(np.frombuffer(data,np.uint8),cv2.IMREAD_COLOR)
    if decoded is None or decoded.size==0: raise ValueError('Undecodable source image: '+rel)
    dst=EXT/cap/rel; sha=hashlib.sha256(data).hexdigest(); changed=False
    def check(): return dst.exists() and dst.stat().st_size==len(data) and digest(dst)==sha
    if not retry(check):
        atomic_bytes(dst,data); changed=True
    return {'path':rel,'sha256':sha,'bytes':len(data),'height':decoded.shape[0],'width':decoded.shape[1],
            'repaired':changed,'mtime_ns':dst.stat().st_mtime_ns}

def image_pass(label,cap,nparts,frames,cams,ann):
    ids,lookup,mv,kv,kp=ann
    folder=OUT/cap
    journal=folder/'verified_images.jsonl'
    previous={}
    if journal.exists():
        for line in retry(journal.read_text).splitlines():
            try:
                r=json.loads(line); previous[r['path']]=r
            except (ValueError,KeyError): pass
    rels=archive_rels(cap,'Mugsy_cameras',STEM,nparts)
    extension=None; extension_info=None
    if label=='HIR112':
        # Only finish the single image cut by the existing ac boundary.
        # A bounded 8 MiB range of ad is enough for normal PNG size; never acquire all 42 missing shards.
        extension=WORK/'HIR_boundary_8MiB.bin'
        tail_rel='Mugsy_cameras/'+STEM+'ad'
        download(BASE+'/'+cap+'/'+tail_rel,extension,byte_range=(0,8*1024**2-1))
        extension_info={'source':tail_rel,'range':[0,8*1024**2-1],'sha256':digest(extension),'purpose':'finish image at acquired-prefix boundary'}
        persist_file(extension,folder/'HIR_boundary_8MiB.bin')
    raw,tar=tar_open(cap,rels,extension)
    rows=[]; pending=[]; seen=set(); overlays={}; started=time.monotonic(); count=0; repaired=0
    errors=[]; end_reason='complete_archive'; boundary_finished=False
    f=open(journal,'a',encoding='utf-8')
    pool=ThreadPoolExecutor(max_workers=4)
    def consume(future,rel,fid,cam,header_extended):
        nonlocal count,repaired
        rec=future.result(); count+=1; repaired+=int(rec.get('repaired',False))
        i=lookup.get(fid)
        reasons=[]
        if i is None: reasons.append('frame_not_in_official_segment_list')
        if cam not in cams: reasons.append('camera_missing')
        if i is not None and not mv[i].any(): reasons.append('no_valid_MANO_target')
        record=dict(rec,frame_id=fid,camera_id=cam,segment=frames.get(fid,''),annotation_index=i,
                    mano_left=bool(mv[i,0]) if i is not None else False,mano_right=bool(mv[i,1]) if i is not None else False,
                    kp_left=bool(kv[i,0]) if i is not None else False,kp_right=bool(kv[i,1]) if i is not None else False,
                    usable=not reasons,issues=reasons)
        rows.append(record); f.write(json.dumps(record)+'\n')
        if count%200==0: f.flush()
        STATUS.update(phase='verify/extract '+label,images=count,repaired=repaired)
        if i is not None and cam in cams and (cam not in overlays) and kv[i].any(): overlays[cam]=(record,kp[i].copy(),kv[i].copy())
    try:
        for member in tar:
            if not member.isfile(): continue
            rel=valid_path(member.name)
            if rel is None or '/images/' not in '/'+rel or Path(rel).suffix.lower() not in ('.png','.jpg','.jpeg'): continue
            # Stop after the first complete image requiring continuation beyond the existing prefix.
            # read-ahead can mark extension slightly early, but we additionally require >=8581 images.
            p=PurePosixPath(rel); cam=p.parent.name
            if not p.stem.isdigit(): raise RuntimeError('Unrecognized image frame: '+rel)
            fid=int(p.stem)
            if rel in seen: raise RuntimeError('Duplicate archive image: '+rel)
            seen.add(rel)
            data=tar.extractfile(member).read()
            if len(data)!=member.size: raise IOError('Incomplete image member: '+rel)
            prior=previous.get(rel); dst=EXT/cap/rel
            skip=False
            if prior:
                try: skip=dst.stat().st_size==prior['bytes'] and dst.stat().st_mtime_ns==prior['mtime_ns'] and hashlib.sha256(data).hexdigest()==prior['sha256']
                except (OSError,KeyError): pass
            if skip:
                from concurrent.futures import Future
                future=Future(); future.set_result(dict(prior,repaired=False))
            else: future=pool.submit(verify_write_image,(cap,rel,data))
            pending.append((future,rel,fid,cam,raw.used_extension))
            if len(pending)>=8: consume(*pending.pop(0))
            if label=='HIR112' and raw.used_extension and len(seen)>=8581:
                boundary_finished=True; end_reason='acquired_prefix_plus_completed_boundary_image'; break
        for job in pending: consume(*job)
        pending=[]
        if label=='HIR112' and not boundary_finished: raise RuntimeError('HIR boundary not completed within 8 MiB allowance')
    finally:
        pool.shutdown(wait=True); f.flush(); os.fsync(f.fileno()); f.close(); tar.close(); raw.close()
    if not rows: raise RuntimeError('No image members found')
    # Build temporal links strictly within capture/camera/segment; preserve frame gaps explicitly.
    groups=defaultdict(list)
    for r in rows:
        if r['usable']: groups[(r['camera_id'],r['segment'])].append(r)
    for group in groups.values():
        group.sort(key=lambda r:r['frame_id'])
        for j,r in enumerate(group):
            r['prev_frame_id']=group[j-1]['frame_id'] if j else None
            r['next_frame_id']=group[j+1]['frame_id'] if j+1<len(group) else None
            r['frame_gap']=r['frame_id']-group[j-1]['frame_id'] if j else None
    # JSONL retains every checked image, including explicit reasons if a source label is unavailable.
    atomic_bytes(folder/'all_images_audit.jsonl', ''.join(json.dumps(r)+'\n' for r in rows).encode())
    usable=[r for r in rows if r['usable']]
    atomic_bytes(folder/'trainable_manifest.jsonl',''.join(json.dumps(r)+'\n' for r in usable).encode())
    seq=[{'camera_id':cam,'segment':seg,'frame_ids':[r['frame_id'] for r in group]} for (cam,seg),group in sorted(groups.items())]
    save_json(folder/'temporal_sequences.json',seq)
    # Camera-frame projection statistics and visual evidence, with real calibration.
    proj={}; tiles=[]
    for cam,(r,world,valid) in sorted(overlays.items()):
        im=cv2.imread(str(EXT/cap/r['path']))
        c=cams[cam]; R=np.asarray(c['R']); t=np.asarray(c['t']).reshape(3)
        focal=np.asarray(c['focal']); pp=np.asarray(c['princpt'])
        stats=[]
        for h in range(2):
            if not valid[h]: continue
            xyz=world[h]@R.T+t; z=xyz[:,2]
            uv=xyz[:,:2]/np.maximum(z[:,None],1e-6)*focal+pp
            inside=(z>0)&(uv[:,0]>=0)&(uv[:,0]<im.shape[1])&(uv[:,1]>=0)&(uv[:,1]<im.shape[0])
            stats.append({'hand':['left','right'][h],'positive_depth':float((z>0).mean()),'inside_image':float(inside.mean())})
            for x,y in uv[inside]: cv2.circle(im,(int(x),int(y)),4,((0,255,0) if h==0 else (0,0,255)),-1)
        proj[cam]=stats
        cv2.putText(im,label+' '+cam+' '+str(r['frame_id']),(10,30),cv2.FONT_HERSHEY_SIMPLEX,.7,(255,255,255),2)
        tiles.append(cv2.resize(im,(334,512)))
    if tiles:
        montage=np.concatenate(tiles,axis=1); ok,png=cv2.imencode('.png',montage); assert ok
        atomic_bytes(folder/'projection_overlay.png',png.tobytes())
    missingkp=sum(1 for r in usable if not (r['kp_left'] and r['kp_right']))
    summary={'label':label,'capture':cap,'images_verified':len(rows),'images_trainable_MANO':len(usable),
      'images_repaired_or_written':repaired,'images_with_some_missing_keypoint_targets':missingkp,
      'image_bytes':sum(r['bytes'] for r in rows),'sequences':len(seq),'segments':len({r['segment'] for r in usable}),
      'source_end':end_reason,'extension':extension_info,'projection_samples':proj,
      'unusable_images':[{k:r[k] for k in ('path','issues')} for r in rows if not r['usable']],
      'elapsed_seconds':round(time.monotonic()-started,1)}
    save_json(folder/'quality_report.json',summary)
    say(label+' RESULT '+json.dumps({k:summary[k] for k in ('images_verified','images_trainable_MANO','images_repaired_or_written','images_with_some_missing_keypoint_targets')}))
    return summary

results=[]
try:
    say('Repair scope: existing 3 captures; CPU; no training; resumable logs on Drive.')
    # Source checksums replace stale sidecar claims. BKS an is repaired during this first pass.
    for label,cap,nparts in CAPS:
        STATUS.update(phase='metadata '+label,images=0)
        frames,cams=metadata(cap)
        ann=labels(cap,frames)
        result=image_pass(label,cap,nparts,frames,cams,ann)
        results.append(result)
        save_json(OUT/'repair_progress.json',{'captures':results,'downloaded_bytes_this_run':NEW_BYTES,'complete':False})
        del ann
    total=sum(r['images_verified'] for r in results)
    ready=sum(r['images_trainable_MANO'] for r in results)
    report={'captures':results,'verified_images':total,'MANO_trainable_images':ready,
      'all_acquired_images_have_MANO_training_targets':ready==total,
      'downloaded_bytes_this_run':NEW_BYTES,'elapsed_seconds':time.monotonic()-START,
      'scope':'All complete image members in acquired 202107 and BKS archives, and acquired HIR aa-ac prefix plus completed boundary image.',
      'assumptions':['Temporal order uses official frame IDs within capture/camera/segment; frame gaps retained; no guessed FPS.',
        'Original 24-joint keypoint order retained; world keypoints and calibration translation are mm.',
        'MANO pose is 48 axis-angle parameters, shape has 10 values; translation retained in source units.',
        'Missing keypoints are masked; finite MANO targets can still supervise these images.',
        'This checks data integrity, joins and projection samples; it does not prove every published fit is geometrically perfect.',
        'The full unacquired HIR capture is outside scope. No model training performed.']}
    save_json(OUT/'FINAL_QUALITY_REPORT.json',report)
    say('FINAL: %d/%d image records have valid MANO supervision; downloaded %.3f GiB.'%(ready,total,NEW_BYTES/GIB))
    say('Inspect FINAL_QUALITY_REPORT.json and projection_overlay.png in each capture folder.')
    if ready!=total: raise RuntimeError('Unmatched source images remain: see explicit unusable_images. NOT claiming 100% ready.')
    say('INTEGRITY + MANO LABEL JOIN COMPLETE. Await independent Drive persistence/overlay verification.')
except BaseException as e:
    say('STOPPED SAFELY: '+type(e).__name__+': '+str(e))
    try: save_json(OUT/'last_interruption.json',{'error':repr(e),'status':STATUS,'completed_captures':results,'time':time.strftime('%Y-%m-%dT%H:%M:%S')})
    except Exception: pass
    raise
finally:
    HALT.set()
